# W1D1｜PyTorch 张量操作 + 自动求导

> 学习日期：2026-04-10
> 目标：掌握 PyTorch 核心 API，理解自动求导机制，夯实 Day 1 基础

# 一、张量（Tensor）— 最底层数据结构

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 创建方式 | `torch.tensor()` vs `torch.Tensor()` | 两者区别（是否拷贝数据、类型推断） |
| 数据类型 | `float/int/bool/dtype` | 如何指定 device + dtype 同时创建 |
| GPU 迁移 | `.cuda()` / `.to(device)` | 如何判断 GPU 可用 |
| 形状操作 | `view / reshape / transpose / permute` | view vs reshape 区别（是否连续） |
| 索引切片 | `tensor[mask]` / `torch.masked_select` | 如何按条件筛选 |
| 运算 | `matmul / mm / @` / `torch.sum / mean / max` | 矩阵乘法哪个最快/最安全 |
| 与 NumPy 互转 | `tensor.numpy()` / `torch.from_numpy(nparr)` | 共享内存问题 |

---

## 1.1 创建方式详解

## 四类方法参数对比

| 方法 | device 参数 | requires_grad | dtype 参数 | 数据来源 | 是否拷贝 |
|---|---|---|---|---|---|
| `torch.Tensor(data)` | ❌ 无 | ❌ 无 | ❌ 无（固定float32） | 传入已有数据 | ✅ 拷贝 |
| `torch.tensor(data)` | ✅ 有 | ✅ 有 | ✅ 有（自动推断） | 传入已有数据 | ✅ 拷贝 |
| `torch.from_numpy(arr)` | ❌ 无 | ❌ 无 | ❌ 无（跟随numpy） | 必须是numpy数组 | ❌ 共享内存 |
| `torch.zeros()` / `torch.ones()` | ✅ 有 | ✅ 有（默认False） | ✅ 有（默认float32） | 自己生成 | ✅ 独立 |

## torch.Tensor() — 类构造函数（不推荐）

实际上是 `torch.FloatTensor` 的别名，类型固定 float32，无其他参数。

In [ ]:
torch.Tensor([1, 2, 3])  # → [1., 2., 3.] float32
t = torch.Tensor([1., 2.]).to('cuda').requires_grad_(True)

## torch.tensor() — 工厂函数（推荐）

自动推断类型，拷贝数据，参数最全。

In [ ]:
torch.tensor([1, 2, 3])                            # int64
torch.tensor([1., 2., 3.])                         # float32
torch.tensor([1, 2], dtype=torch.float32)          # 强制 float32
x = torch.tensor([1., 2., 3.], device='cuda', requires_grad=True)  # 一步到位

## torch.from_numpy() — 共享内存（需注意坑）

必须传 numpy 数组，共享内存修改互相影响。

In [ ]:
import numpy as np
arr = np.array([1., 2., 3.])
t = torch.from_numpy(arr)  # 共享内存！
t[0] = 99.
print(arr)  # [99., 2., 3.] ← numpy 数组被改了！

dtype 跟随 numpy，不统一成 PyTorch 默认值。

## torch.zeros() / torch.ones() — 初始化

In [ ]:
torch.zeros(3, 4, dtype=torch.float32, device='cuda', requires_grad=True)
torch.ones(2, dtype=torch.int64)

## 默认值

所有方法不指定时 → **device=cpu，requires_grad=False**

---

## 1.2 形状操作

### stride 详解

stride 是"跳读规则"，决定按当前维度顺序读取 storage 时，每个维度内跳几个元素才能读完。

In [ ]:
x = torch.randn(2, 3, 4)  # shape: (2, 3, 4)
x.stride()  # → (12, 4, 1)

把 `(2, 3, 4)` 想象成 2 页纸，每页 3 行，每行 4 个元素。

### 连续 stride 的计算公式

In [ ]:
stride[i] = shape[i+1] × shape[i+2] × ... × shape[last]

## view

**本质**：换一套 shape/stride 来解释同一块 storage，数据完全不动。

In [ ]:
x = torch.randn(2, 3, 4)  # (2, 3, 4), stride (12, 4, 1)
x.view(6, 4)   # (6, 4), stride (4, 1) — 数据完全不动
x.view(24,)    # (24,)
x.view(1, 2, 3, 4)  # 任意维度都可以，只要 1×2×3×4 = 24
x.view(-1)     # (24,) — -1 自动推断

view 可以任意维度，不限于二维。要求：tensor 必须连续，不连续时报错：

In [ ]:
x_t = x.transpose(0, 1)  # stride (4, 12, 1) — 不连续
x_t.view(24)   # ❌ RuntimeError

## transpose

**本质**：交换两个维度的位置，数据不动，只换 shape 和 stride。

In [ ]:
x = torch.randn(2, 3)        # shape (2, 3), stride (3, 1)
x_t = x.transpose(0, 1)     # shape (3, 2), stride (1, 3)

**transpose 后 stride 的算法：直接交换对应位置的值。**

In [ ]:
x = torch.randn(2, 3, 4)           # stride (12, 4, 1)
x.transpose(0, 1)                   # stride (4, 12, 1)

transpose 后 tensor 必然不连续，因为 stride 顺序和 storage 物理顺序对不上了。

## reshape

**本质**：等价于 `contiguous().view()`。

In [ ]:
x.reshape(6, 4)  # 连续 tensor → 直接 view，不复制
x_t.reshape(6,)  # 不连续 tensor → 先复制重排，再 view

**优先用 `reshape`**，永远不报错。

## permute

入参语义：新维度 i 来自旧维度 dim_i。

In [ ]:
x = torch.randn(2, 3, 4)  # shape: (2, 3, 4)
x.permute(2, 0, 1)  # shape: (4, 2, 3)

和 transpose 完全一致——按同样索引直接重新排列 stride。

## squeeze / unsqueeze

In [ ]:
x = torch.randn(1, 3, 1, 8, 1)  # 5个维度
x.squeeze().shape   # (3, 8) — 删除所有 size=1 的维度
x.squeeze(0).shape  # (3, 1, 8, 1) — 删最外层
x.unsqueeze(0).shape   # (1, 2, 3) — 在最前面插

## flatten

In [ ]:
x = torch.randn(2, 3, 4)  # (2, 3, 4)
x.flatten().shape           # (24,) — 所有维度展平
x.flatten(1, 2).shape       # (2, 12) — 只展平中间两个维度

等价于 `reshape(-1)` 或 `reshape(..., -1)`。

## 三者对比

| | view | transpose | reshape |
|---|---|---|---|
| 数据动了吗？ | ❌ 没动 | ❌ 没动 | ✅ 不连续时会复制重排 |
| 连续性 | 要求连续 | 必然不连续 | 永远成功 |

---

## 1.3 索引切片

### 基础索引 + 切片

In [ ]:
x = torch.randn(2, 3, 4)
x[0].shape         # (3, 4) — 取第一个样本
x[0, 1].shape      # (4,) — 取特定元素
x[0:2].shape       # (2, 3, 4) — 切片
x[:, 1:].shape     # (2, 2, 4) — 跨维度切片

### None（等价于 unsqueeze）

In [ ]:
x = torch.randn(2, 3)
x[None, :].shape   # (1, 2, 3) — 等价于 x.unsqueeze(0)
x[:, None].shape   # (2, 1, 3)

### view vs copy 的分界线（核心！）

**返回 view**（共享底层 storage）：基础切片索引、`None` 索引、`squeeze`/`unsqueeze`、`transpose`/`permute`、`view`（连续时）、`expand`

**返回 copy**（独立新 storage）：`clone()`、布尔掩码索引、整数数组索引、`torch.masked_select`

In [ ]:
x = torch.randn(2, 3)
y = x[0:1]           # view
z = x[x > 0]          # copy
print(x.storage().data_ptr() == y.storage().data_ptr())  # True → view
print(x.storage().data_ptr() == z.storage().data_ptr())  # False → copy

### 布尔掩码

In [ ]:
x = torch.randn(3, 4)
mask = x > 0
x[mask].shape              # (?,) — 展平成一维
torch.masked_select(x, mask).shape  # 同上，完全等价

布尔掩码返回 **copy**，不共享数据。

### 整数数组索引

In [ ]:
x = torch.randn(5, 3)
rows = torch.tensor([0, 2, 3])
x[rows].shape        # (3, 3) — 取指定行
x[rows, 1].shape     # (3,) — 取这些行的第二列

整数数组索引返回 **copy**，不共享数据。

### 分步法：不连续行列组合

In [ ]:
x = torch.randn(5, 3)
rows = torch.tensor([0, 2, 3])
cols = torch.tensor([0, 2])
x[rows][:, cols].shape  # (3, 2) — 分两步取不连续行列

---

## 1.4 常用运算

### 逐元素运算

In [ ]:
x + 1      # 加减乘除基本运算
x ** 2     # 幂运算
torch.sqrt(x)  # 开方
x.abs().log()  # 链式调用

### 归约运算

In [ ]:
x.sum()              # 全部求和 → scalar
x.sum(dim=0)        # 按 dim=0 求和 → (3,)
x.mean()             # 均值
x.max()              # 最大值 → scalar
x.argmax()            # 最大值索引（flatten后）
x.max(dim=0)         # 返回 (values, indices)

### 矩阵运算

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(4, 5)
(a @ b).shape   # (3, 5)

**推荐 `@` / `matmul`**，支持多维 + 广播，是全能版。

### 广播机制

从左补1，从右对齐检查。

In [ ]:
x = torch.randn(3, 4)
y = torch.randn(4)       # 1D
x + y  # y补成(1,4) → broadcast到(3,4) ✅

### torch.where

In [ ]:
torch.where(x > 0, x, 0)  # x>0保留x，否则填0（限幅）

### in-place 操作

下划线 `_` 结尾 = in-place，直接修改原 tensor。

In [ ]:
x.add_(1)           # x = x + 1，原地改
x.zero_()           # 全部变成 0

---

## 1.5 GPU 迁移与设备

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.randn(10, device=device)
model = model.to(device)

---

## 1.6 与 NumPy 互转

In [ ]:
arr = x.numpy()              # Tensor → NumPy（共享内存时有问题）
x = torch.from_numpy(arr)    # NumPy → Tensor（共享内存）
x = tensor.numpy()           # 需要 requires_grad=False 或 detach

---

## 1.7 默认值总结

| 方法 | device | requires_grad | dtype |
|---|---|---|---|
| `torch.Tensor()` | cpu | False | float32 |
| `torch.tensor()` | cpu | False | 自动推断 |
| `torch.zeros/ones()` | cpu | False | float32 |
| `torch.from_numpy()` | cpu | False | 跟随numpy |

---

## 代码练习

In [ ]:
import torch
import numpy as np

# 1. torch.Tensor() vs torch.tensor() — 类型推断差异
t1 = torch.Tensor([1, 2, 3])       # 固定 float32，值变成 1.0, 2.0, 3.0
t2 = torch.tensor([1, 2, 3])       # 自动推断 int64

# 2. 拷贝验证
lst = [1., 2., 3.]
t = torch.Tensor(lst)
t[0] = 99.
print("原始 list 不变:", lst)  # 不受影响

# 3. from_numpy() 共享内存
arr = np.array([1., 2., 3.])
t_share = torch.from_numpy(arr)
t_share[0] = 99.
print("from_numpy 改了 numpy:", arr)  # [99., 2., 3.]

# 4. view vs reshape
x = torch.randn(2, 3)
x_t = x.transpose(0, 1)
try:
    x_t.view(6)
except RuntimeError as e:
    print("view 不连续报错:", e)
print("reshape 不连续:", x_t.reshape(6))

# 5. stride 验证
x = torch.randn(2, 3, 4)
print("原始 stride:", x.stride())  # (12, 4, 1)

# 6. 连续性判断
x = torch.randn(2, 3)
print("is_contiguous:", x.is_contiguous())
x_t = x.transpose(0, 1)
print("transpose 后 is_contiguous:", x_t.is_contiguous())

# 二、自动求导（Autograd）— PyTorch 灵魂

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| `requires_grad` | 默认为 False，设为 True 开启追踪 | 哪些操作会默认开启 |
| `backward()` | 反向传播计算梯度 | 何时调用，梯度会累加还是覆盖 |
| `grad` | 保存梯度值 | 多个 `backward()` 时梯度如何变化 |
| `grad_fn` | 记录创建张量的运算 | 用于什么 |
| `torch.no_grad()` | 前向推理时不追踪梯度 | 与 `eval()` 区别 |
| `detach()` | 截断计算图 | 何时需要 detach |
| `hook` 机制 | 注册前向/反向 hook | 有什么用 |

**⚠️ 面试必答题：**
- PyTorch 反向传播原理？计算图是怎么构建的？
- `backward()` 掉了梯度会怎样？多次 backward 梯度累加还是覆盖？
- 梯度消失/爆炸的原因？在 PyTorch 中如何检测和解决？

---

# 1-5 自动求导（Autograd）

## 1-5-1 计算图与反向传播原理

PyTorch 的自动求导基于**动态计算图**（Dynamic Computation Graph）。

**核心概念**：
- 每当对 `requires_grad=True` 的张量进行运算，PyTorch 会构建一个反向传播的 DAG
- 叶子节点（leaf nodes）是用户直接创建的张量（通过 `torch.tensor()` 等）
- 根节点（root）是最终输出标量（scalar），通常是一个 loss

---

In [ ]:
import torch

# 叶子节点：用户创建，requires_grad 默认为 False
x = torch.tensor([1., 2., 3.], requires_grad=True)  # 叶子
y = x ** 2          # 中间节点，grad_fn 记录运算
z = y.sum()         # 根节点，backward 从这里开始

print("x 是叶子:", x.is_leaf)
print("y 是叶子:", y.is_leaf)
print("z 是叶子:", z.is_leaf)
print("z.grad_fn:", z.grad_fn)  # <SumBackward0>

---

**DAG 的构建是动态的**：每次前向传播都会重新构建计算图，修改代码后图结构随之变化。

---

In [ ]:
# 同一个变量两次前向，建两个图
a = torch.tensor([1., 2.], requires_grad=True)
b = a ** 2
c = b * 2
d = c.sum()
print(d.grad_fn)  # <AddBackward1> — 两个图各自独立

---

### 动态图 vs 静态图

| | PyTorch（动态图） | TensorFlow 1.x（静态图） |
|--|---|---|
| 建图时机 | 前向传播时同步构建 | 先定义图，后执行 |
| 控制流 | 直接用 Python `if/for/while` | 需要 `tf.cond`/`tf.while_loop` |
| 调试 | 直接报错，可 print | 需要 `sess.run` 才能看值 |
| 灵活性 | 极高，代码怎么写图就怎么建 | 需预先定义所有分支 |

**动态图优势**：代码即模型，不用预先声明图结构。

**静态图优势**：图结构已知，可做全局优化（如算子融合、内存规划），性能更好。TensorFlow 2.x 默认 eager execution（动态图），但用 `tf.function` 可以 jit 编译成静态图加速。PyTorch 的 `torch.compile` 也是类似思路——先把动态图"冻结"成静态图再优化。

---

In [ ]:
# 动态图：每行代码立即执行
x = torch.tensor([1., 2.])
y = x * 2        # 立即计算
z = y + 1        # 立即计算

# 静态图（TensorFlow 风格）：
# x = tf.placeholder(tf.float32)   # 先声明
# y = x * 2                        # 只是画边，不计算
# with tf.Session() as sess:
#     result = sess.run(y, feed_dict={x: [1., 2.]})  # 实际执行

---

**Notebook 中的内存问题**：动态图下，每次 cell 跑前向都会创建新的计算图节点，如果不断保存中间结果（图节点被引用），内存会累积。定期 `del` 不需要的变量 + `gc.collect()` 可缓解。静态图因为图结构固定，不存在这个问题。

---

## 1-5-2 backward() 与梯度计算

`backward()` 从当前张量反向传播到所有叶子节点，计算 `∂tensor/∂leaf`。

**必须条件**：调用 `backward()` 的张量必须是**标量**（scalar，形状为空），或者传入 `gradient` 参数。

---

In [ ]:
import torch

# y = x²，求 dy/dx
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2       # y = [1, 4, 9]
z = y.sum()      # z = 14，必须是标量才能直接 backward()

z.backward()     # 等价于 z.backward(torch.tensor(1.))
print(x.grad)    # tensor([2., 4., 6.]) — dy/dx = 2x

# 验证：x=[1,2,3]，2x = [2,4,6] ✓

---

**非标量如何 backward**：需要传入 `gradient` 参数（与输出同形状），表示链式求导的起点。

---

In [ ]:
# J = [J₁, J₂]，每个 Jᵢ 对 x 求偏导
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2       # y = [1, 4, 9]
# y 有 3 个输出，每个都对 x 求偏导构成雅可比矩阵

v = torch.tensor([1., 0., 0.])  # 只想求 J₁ 对 x 的偏导
y.backward(v)
print(x.grad)     # tensor([2., 0., 0.]) = ∂J₁/∂x = 2x

---

---

## 1-5-2.5 标量 backward vs 向量 backward（gradient 参数详解）

### 核心问题：为什么非标量不能直接 backward？

标量 `loss.backward()` 直接能跑，是因为 PyTorch 知道"从 loss（一个数）往回传梯度，起点就是 1"。

但如果 `y` 是向量，`y.backward()` 会遇到歧义：**y 有多个元素，每个元素对上游参数的梯度都不一样**。这时候 PyTorch 不知道该把什么值传回去，所以直接报错：`grad can be implicitly created only for scalar outputs`。

### 用具体例子理解三种梯度

以一个完整的前向计算图为例：

```
x → [matmul] → y → [减法] → [平方] → [求平均] → loss
                                              ↑
                                         y_true
```

**已知数值**：
```
x = [1, 2]
w = [[1, 3], [2, 4]]
y_true = [10, 20]
```

**前向传播（逐步计算）**：

| 步骤 | 计算 | 结果 |
|------|------|------|
| y = x @ w | `y[0]=1×1+2×2, y[1]=1×3+2×4` | y = [5, 11] |
| diff = y - y_true | `5-10, 11-20` | diff = [-5, -9] |
| square = diff² | `(-5)², (-9)²` | square = [25, 81] |
| loss = mean(square) | `(25+81)/2` | loss = 53 |

**反向传播第一步：dloss/dy**

loss 对 y 求偏导——loss 是标量，y 是向量，所以 dloss/dy 也是向量：

```
loss = ((y[0]-10)² + (y[1]-20)²) / 2

∂loss/∂y[0] = 2×(y[0]-10) / 2 = y[0] - 10 = 5 - 10 = -5
∂loss/∂y[1] = 2×(y[1]-20) / 2 = y[1] - 20 = 11 - 20 = -9

所以 dloss/dy = [-5, -9]  ← 就是 diff 本身！
```

**反向传播第二步：dy/dw（雅可比矩阵）**

y 对 w 求偏导——y 是向量，w 是 2×2 矩阵，所以 dy/dw 是一个 2×2 的雅可比矩阵：

```
y[0] = x[0]×w[0,0] + x[1]×w[1,0] = 1×w[0,0] + 2×w[1,0]
y[1] = x[0]×w[0,1] + x[1]×w[1,1] = 1×w[0,1] + 2×w[1,1]

∂y[0]/∂w[0,0] = 1    ∂y[0]/∂w[0,1] = 0
∂y[0]/∂w[1,0] = 2    ∂y[0]/∂w[1,1] = 0
∂y[1]/∂w[0,0] = 0    ∂y[1]/∂w[0,1] = 1
∂y[1]/∂w[1,0] = 0    ∂y[1]/∂w[1,1] = 2

dy/dw = [[1, 0],   ← y[0] 对 w 的偏导
         [2, 0],
         [0, 1],   ← y[1] 对 w 的偏导
         [0, 2]]
```

注意：y[i] 只和 w[:, i]（第 i 列）有关，和 w[:, 1-i] 无关，所以每列只有一个元素有值。

**反向传播第三步：dloss/dw = dloss/dy · dy/dw（链式法则）**

链式法则公式：`∂loss/∂w[i,j] = ∂loss/∂y[j] × ∂y[j]/∂w[i,j]`

| | 第 0 列 (j=0) | 第 1 列 (j=1) |
|--|---------------|---------------|
| 第 0 行 (i=0) | `(-5) × 1 = -5` | `(-9) × 1 = -9` |
| 第 1 行 (i=1) | `(-5) × 2 = -10` | `(-9) × 2 = -18` |

```
dloss/dw = [[-5, -9],
             [-10, -18]]
```

### 标量 backward vs 向量 backward 的本质区别

| | `loss.backward()` | `y.backward(gradient)` |
|--|--|--|
| 起点梯度 | 自动假设 = 1（标量的上游梯度就是 1） | 你手动指定 upstream |
| 传播范围 | 从当前节点一直传到最后（叶子节点） | 只到当前节点为止 |
| 典型场景 | 训练神经网络算 loss 梯度 | 策略梯度、手动梯度构造 |

**`y.backward(gradient=[1,1])` 的实际含义**：

```
y → [matmul] → ... → loss
↑
手动传入 [1,1]，跳过 y 之后的所有计算
```

传入 `[1,1]` 相当于告诉 PyTorch："从 y 往上游传的时候，就传 [1,1] 就行了，不需要再算 loss → y 那一段的梯度了"。

也就是说：**把本应该从 loss 计算到 y 时产生的梯度（-5, -9），替换成你手动输入的值（1, 1）**。

---

In [ ]:
# 等价于：手动替换了 dloss/dy 本来的值 [-5, -9]
y.backward(gradient=torch.tensor([1.0, 1.0]))
# dloss/dw[j,k] = upstream[k] × x[j]
# 结果 = [[1, 1], [2, 2]]（而不是 [[-5,-9], [-10,-18]]）

---

---

## 1-5-3 梯度累加机制（关键！）

**默认行为：梯度是累加的，不是覆盖。**

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)

# 第一次 backward
y = (x ** 2).sum()
y.backward()
print("第1次:", x.grad)   # tensor([2., 4., 6.])

# 第二次 backward — 梯度会累加！
y2 = (x ** 2).sum()
y2.backward()
print("第2次:", x.grad)   # tensor([4., 8., 12.]) — 累加了！

# 正确的多步训练：每次前向之间清梯度
x.grad.zero_()           # 原地清零
y3 = (x ** 2).sum()
y3.backward()
print("清零后:", x.grad)   # tensor([2., 4., 6.]) — 正确

---

**训练循环中的标准写法**：

---

In [ ]:
for data, target in dataloader:
    optimizer.zero_grad()   # Step 1: 清梯度
    output = model(data)     # Step 2: 前向
    loss = criterion(output, target)
    loss.backward()          # Step 3: 反向
    optimizer.step()         # Step 4: 更新参数

---

**为什么设计成累加而不是覆盖？**

这是**有意为之**，而不是"忘了清零"。累加设计是为了支持两种场景：

**场景1：梯度累积（大 batch 训练）**

---

In [ ]:
# batch_size=64 显存不够，用4个 batch_size=16 累积
for i, (data, target) in enumerate(dataloader):
    output = model(data)
    loss = criterion(output, target) / 4  # 归一化
    loss.backward()   # 累加到 .grad
    
    if (i + 1) % 4 == 0:
        optimizer.step()      # 用累积的梯度更新
        optimizer.zero_grad()  # 4个batch累积完，清零

---

**场景2：多 loss 多路径回传**

---

In [ ]:
# 一个 shared 参数被多个分支使用
shared_params = ...
out1 = branch1(shared_params)
out2 = branch2(shared_params)

loss1 = criterion(out1, target1)
loss2 = criterion(out2, target2)

loss1.backward()  # shared_params.grad 有了第一份梯度
loss2.backward()  # 累加第二份，两个分支的梯度都被保留

---

**如果设计成"清零覆盖"**：上述两个场景都直接坏掉——要么无法做梯度累积，要么多路径的梯度互相覆盖。累加设计把控制权交给用户，框架不隐式做主。

---

In [ ]:
# 梯度累积：模拟 batch_size=64，实际用两个 batch_size=32
model.zero_grad()           # 清零
loss1 = model(batch1).sum() / 2  # 32
loss1.backward()             # 梯度 /2 累加
loss2 = model(batch2).sum() / 2  # 32
loss2.backward()             # 梯度 /2 累加
# 等效于 batch_size=64 的梯度

---

---

## 1-5-4 no_grad / set_grad_enabled / eval

### torch.no_grad()

**作用**：禁用梯度计算，不构建计算图，节省显存和计算量。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

with torch.no_grad():
    z = y * 2      # 这里不追踪梯度
    print("no_grad 中:", z.requires_grad)  # False

print("no_grad 外:", z.requires_grad)  # False — 不影响外面的变量

# 用于推理/评估阶段
model.eval()
with torch.no_grad():
    predictions = model(test_data)

---

**另一个写法**：`@torch.no_grad()` 装饰器，效果相同。

---

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    return model(data)

---

### torch.set_grad_enabled()

**作用**：动态切换梯度追踪状态。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)

torch.set_grad_enabled(False)
y1 = x * 2
print("关闭后:", y1.requires_grad)  # False

torch.set_grad_enabled(True)   # 重新开启
y2 = x * 2
print("开启后:", y2.requires_grad)  # True

---

### model.eval() vs model.train()

| 模式 | 作用 | 影响 |
|---|---|---|
| `train()` | 训练模式 | BatchNorm 用 batch 统计量，Dropout 生效 |
| `eval()` | 评估模式 | BatchNorm 用全局统计量（moving average），Dropout 不生效 |

---

In [ ]:
model.train()
for batch in train_loader:
    optimizer.zero_grad()
    loss = ...
    loss.backward()

model.eval()                    # 切换
with torch.no_grad():
    for batch in val_loader:
        ...

---

**注意**：`eval()` 不等于 `no_grad()`！两者针对不同问题，要叠加使用：

---

In [ ]:
model.eval()
with torch.no_grad():           # 双重保险
    val_loss = ...

---

---

## 1-5-5 detach() 截断计算图

**作用**：将张量从计算图中分离出来，创建一个**共享存储但不共享梯度追踪**的新张量。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2                # y 仍在图中
z = y.detach()            # z 与 y 共享数据，但脱离梯度追踪

print("z.requires_grad:", z.requires_grad)  # False
print("y.requires_grad:", y.requires_grad)  # True

# z 可以正常计算，但不会影响 x 的梯度
z_new = z * 2             # 无梯度追踪

---

### 共享存储的陷阱

**detach() 出来的张量和原张量共享同一块底层内存**，修改其中一个会影响另一个：

---

In [ ]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x * 2      # y = [2, 4, 6]
z = y.detach() # z 和 y 共享底层存储

z[0] = 99      # 原地修改 z
print(y)        # y = [99, 4, 6]  ← y 也被改了！
print(z)        # z = [99, 4, 6]

---

**安全用法**：detach 后立即做只读操作（打印、存列表、画图），不会出问题。

**如果需要修改后不影响原值**，用 `clone()` 深拷贝：

---

In [ ]:
z = y.detach().clone()  # 深拷贝，不共享存储
z[0] = 99
print(y)  # y 不受影响

---

### 典型用途

1. **需要原地修改张量值时**（叶子节点不能直接 in-place 修改）
2. **把张量传给不需要梯度的函数**（如 numpy 操作、打印、画图）
3. **阻止梯度回传到某条分支**（如 Actor-Critic 中 detach baseline）

---

In [ ]:
# 错误示范：直接修改需要梯度的叶子节点
x = torch.tensor([1., 2., 3.], requires_grad=True)
x[0] = 10                   # RuntimeError: a view of a variable

# 正确做法
x_detached = x.detach()
x_detached[0] = 10          # OK

---

**detach() vs no_grad()**：
- `no_grad()`：**整个代码块**都不追踪梯度
- `detach()`：**单个张量**从图中分离出来（但共享存储）

---

## 1-5-6 hook 机制（进阶）

**作用**：在不需要修改前向/反向代码的情况下，**拦截**前向传播或反向传播的过程，查看或修改中间张量/梯度。

**注册层级说明**：hook 绑定的位置决定触发次数：

| 注册位置 | 触发次数 | 说明 |
|---------|---------|------|
| `model`（根模块） | 1次 | 拦截整个模型的最终输入输出 |
| `model.fc1`（子模块） | 1次 | 只在 fc1 算完时触发 |
| `model.fc2`（子模块） | 1次 | 只在 fc2 算完时触发 |

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 5),
    nn.Linear(5, 2)
)

# 注册在根模块：只触发1次（最终输出）
handle = model.register_forward_hook(
    lambda m, inp, out: print(f'根模块 hook，output shape: {out.shape}')
)

# 如果想拦截每个层，需要遍历注册
for name, module in model.named_children():
    module.register_forward_hook(
        lambda m, inp, out: print(f'  层 {name} hook')
    )

x = torch.randn(2, 10)
y = model(x)
# 输出：
#   层 0 hook
#   层 1 hook
#   根模块 hook

handle.remove()

---

### 注册反向 hook

---

In [ ]:
def backward_hook(module, grad_input, grad_output):
    print(f"输入梯度形状: {[g.shape for g in grad_input]}")
    print(f"输出梯度形状: {grad_output[0].shape}")
    return grad_input  # 可以修改输入梯度

handle = model.register_backward_hook(backward_hook)

x = torch.randn(2, 10, requires_grad=True)
y = model(x)
loss = y.sum()
loss.backward()

---

### 常用场景

1. **特征提取**：在不修改模型的情况下，获取中间层激活值
2. **梯度检查**：验证梯度计算是否正确
3. **梯度修改**：在反向传播时人为注入噪声、裁剪等

---

In [ ]:
# 完整例子：提取 ResNet 中间层特征
import torchvision.models as models

model = models.resnet18(pretrained=True)
features = {}

def hook_fn(name):
    def fn(module, input, output):
        features[name] = output.detach()
    return fn

model.layer1.register_forward_hook(hook_fn("layer1"))
model.layer2.register_forward_hook(hook_fn("layer2"))

---

---

## 1-5-7 梯度消失与爆炸

### 原因

多层链式求导时，梯度在反向传播中不断连乘：

- **梯度消失**：|∂L/∂W| < 1，连乘后趋近于 0 → 参数几乎不更新
- **梯度爆炸**：|∂L/∂W| > 1，连乘后趋近于 ∞ → 参数大幅震荡

### 核心机制详解

#### 梯度裁剪（直接修改梯度）

---

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

**原理**：计算所有参数梯度的 L2 范数 `||g||₂ = √(g₁² + g₂² + ...)`，如果超过 `max_norm`，等比例缩放回来。

---

In [ ]:
# 等价逻辑
total_norm = 0.0
for p in model.parameters():
    total_norm += p.grad.norm(2).item() ** 2
total_norm = total_norm ** 0.5

if total_norm > max_norm:
    clip_coef = max_norm / total_norm
    for p in model.parameters():
        p.grad.mul_(clip_coef)  # 直接缩放梯度

---

**特点**：这是**唯一一个直接在反向传播后修改梯度**的方法，属于"事后补救"。只能防止爆炸，不能防止消失。

#### 残差连接（加法保梯度）

```
普通层：output = H(x)
残差层：output = H(x) + x
```

**关键**：加法的反向传播是"分流"的——`∂(H+x)/∂x = ∂H/∂x + 1`。

无论 H(x) 的梯度多小，加个 `+1` 就让梯度永远不会消失。也不存在梯度爆炸的问题（加法不会让梯度相乘变大）。

#### 归一化（稳定激活值，间接稳定梯度）

BatchNorm 前向：

---

In [ ]:
mu = x.mean(dim=0)
var = x.var(dim=0)
x_norm = (x - mu) / sqrt(var + eps)
y = gamma * x_norm + beta

---

**对梯度的影响**：
- **间接**：把激活值压到稳定范围（前向数值稳定 → 反向梯度数值也稳定）
- **直接**：gamma/beta 作为可学习参数，提供稳定的梯度回传路径（形状固定为特征维度，不会随深度指数变化）

#### 激活函数（间接影响梯度）

激活函数在前向时改变激活值分布，间接影响梯度：

| 激活函数 | 前向特点 | 对梯度的影响 |
|---------|---------|-------------|
| Sigmoid | 输出 0~1 | 导数最大 0.25，深层连乘≈0，梯度消失 |
| Tanh | 输出 -1~1 | 导数最大 1，比 Sigmoid 好，但仍有消失问题 |
| ReLU | 负区=0，正区=原值 | 正区导数恒为 1，不消失 |

**ReLU 的梯度（工程定义）**：
```
x > 0: 梯度 = 1
x < 0: 梯度 = 0（截断）
x = 0: 梯度 = 0 或 1（工程规定，不是数学严格定义）
```

**注意**：ReLU 在 x=0 处数学上不可导，但工程上通过**规定 subgradient**（次梯度）让它能参与反向传播。

#### 权重初始化（预防）

权重太小 → 激活值逐层变小 → 梯度消失
权重太大 → 激活值逐层变大 → 梯度爆炸

合理的初始化（如 Kaiming/Xavier）让各层激活值和梯度的方差在合理范围，从源头降低消失/爆炸风险。

#### LSTM/GRU（门控机制）

RNN 循环使用同一个权重矩阵 W：`h_t = h_{t-1} @ W`

梯度回传时：`∂h_T/∂h_t = W^T @ W^T @ ... @ W^T`（T-t 次连乘）

LSTM/GRU 通过**门控**决定保留多少历史、多少新信息：
```
h_t = h_{t-1} * output_gate + new_gate * input_gate
```
梯度可以"抄近道"不经过所有 W 连乘，从根本上缓解消失/爆炸。

### 方法分类总结

| 方法 | 操作阶段 | 机制 |
|------|---------|------|
| 梯度裁剪 | **反向传播后** | 直接修改梯度，治标 |
| 残差连接 | 前向 | 加法+1保梯度，治本 |
| 归一化 | 前向 | 稳定激活值→稳定梯度，治本 |
| 激活函数 | 前向 | 改变激活值分布，间接影响 |
| 权重初始化 | 前向（训练前） | 预防，治本 |
| LSTM/GRU | 前向 | 门控减少连乘，治本 |

**大多数方法都是在"前向阶段预防"梯度问题，只有梯度裁剪是在"反向传播后直接修改梯度"。**

---

## 1-5-7.5 nn.Module 与 nn.Parameter

### 为什么需要 nn.Module？

纯 tensor 的问题：参数需要手动管理。

---

In [ ]:
# 纯 tensor 写法
W = torch.randn(10, 5, requires_grad=True)
b = torch.randn(5, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.1)  # 手动传入参数列表

---

**nn.Module 的核心价值**：

1. **自动参数收集**：`model.parameters()` 把所有带梯度的参数收拢在一起
2. **设备统一管理**：`model.to('cuda')` 一次移动所有参数
3. **封装复用**：把参数和计算逻辑打包成独立模块

---

In [ ]:
# nn.Module 写法
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.randn(out_features))
    
    def forward(self, x):
        return x @ self.weight + self.bias

model = Linear(10, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 自动收集所有参数

---

### nn.Parameter 是什么？

---

In [ ]:
nn.Parameter(tensor) ≈ torch.tensor(..., requires_grad=True) + 自动注册到父 Module

---

| | 普通 tensor | nn.Parameter |
|--|-----------|--------------|
| `requires_grad` | 默认 False | 默认 True |
| 被 `model.parameters()` 收集 | ❌ | ✅ |
| 能被 optimizer 更新 | ❌ | ✅ |

`nn.Parameter` 就是 `requires_grad=True` 的 tensor，外层包了一层"注册"逻辑。本质上还是个 tensor。

### 计算图视角下的 nn.Module

```
nn.Module
    ├── self.weight (nn.Parameter = requires_grad=True 的 tensor)
    ├── self.bias   (nn.Parameter = requires_grad=True 的 tensor)
    └── forward()
            ↓
        构建计算图（和纯 tensor 完全一样）
```

`nn.Module` 本身不参与计算图，它只是 Python 层的组织结构。真正参与计算图的是 `nn.Parameter`——它们本质上是 tensor。`model(x)` 执行时，实际上是在 tensor 层面做运算，构建计算图。**nn.Module 是组织的壳，nn.Parameter 才是参与梯度追踪的实际的 tensor。**

---

## 本节面试重点

**Q1: PyTorch 反向传播原理？计算图是怎么构建的？**
- PyTorch 构建动态有向无环图（DAG），`requires_grad=True` 的张量参与运算时自动记录
- `backward()` 从根节点反向遍历，叶子节点的 `.grad` 累加梯度

**Q2: `backward()` 多次调用，梯度累加还是覆盖？**
- 累加！每次 `backward()` 都会把新梯度加到 `.grad` 上
- 所以训练循环中必须 `optimizer.zero_grad()` 清零

**Q3: `no_grad()` vs `eval()` 区别？**
- `no_grad()`：不构建计算图，节省显存
- `eval()`：BN 用全局统计量，Dropout 不生效
- 推理时两者叠加：`model.eval() + with torch.no_grad()`

**Q4: `detach()` 什么时候用？**
- 需要对张量值做原地修改时（叶子节点不能直接修改）
- 需要把计算图中间结果传给不需要梯度的操作时

**Q5: 梯度消失/爆炸的解决？**
- 梯度裁剪（`clip_grad_norm_`）、残差连接、归一化、ReLU 替代 Sigmoid、权重初始化

---

## 1-5-8 梯度本质详解（数学直觉）

### 梯度是什么？

**梯度的物理含义：瞬时变化率（导数）在多维空间的推广。**

对于一元函数 `y = f(x)`，导数 `f'(x)` 描述的是：x 每增加 1 个单位，y 变化多少。

对于多元函数 `L = f(w₁, w₂, ..., wₙ)`，梯度 `∇L = [∂L/∂w₁, ∂L/∂w₂, ..., ∂L/∂wₙ]` 描述的是：**每个参数 wᵢ 每增加 1 个单位，loss L 变化多少。**

### 梯度为什么能用来更新参数？

**梯度指向的是函数值增大的方向。**

```
L(w) = (w - 5)²，最小值在 w=5
∇L = dL/dw = 2(w - 5)

w=3 时，∇L = -4（负的）
→ 负梯度方向 = 增大方向的反方向
→ w 应该往正方向走（增大）
→ w_new = 3 - lr×(-4) = 3 + lr×4
```

所以：**沿负梯度方向走，函数值下降。** 这就是梯度下降法的核心直觉。

### 链式法则是梯度的核心计算规则

链式法则把复杂函数的梯度拆解为基本运算的梯度组合：

```
L = sin(w²)
设 u = w²，则 L = sin(u)

dL/dw = dL/du · du/dw = cos(u) · 2w = 2w·cos(w²)
```

PyTorch 的 autograd 引擎就是把这个链式法则**自动化执行**：
- 前向传播：真正计算每个节点的数值
- 反向传播：沿着链式法则的路径，把梯度逐级回传

### 梯度的累加本质

梯度是**加性的**——复合函数的梯度等于各部分梯度的和。

```
L = f(g(h(x)))
dL/dx = dL/df · df/dg · dg/dh · dh/dx
```

每一级都在**乘**（链式），最后得到完整梯度。这就是为什么深层网络的梯度容易消失/爆炸——链路上连续乘了很多小于1或大于1的数。

### 为什么梯度是"瞬时"而非"平均"？

```
f(x) = x²，在 x=3 处
- 瞬时导数：f'(3) = 6（切线斜率）
- x从3到5的平均变化率：(25-9)/(5-3) = 8
```

梯度下降用的是瞬时斜率（6），不是平均斜率（8）。这意味着：
- 每一步都假设"当前切线能很好地近似曲线的一小段"
- 步子越小，线性近似的误差越小
- 这也是为什么学习率太大时会发散——步子大到线性近似已经严重失真

### 梯度为负时更新的方向

```
w_new = w - lr × gradient
```

| 梯度方向 | 含义 | 更新动作 | 结果 |
|---|---|---|---|
| 负梯度 | w 增加则 L 减少 | w 增大 | L 减少 ✅ |
| 正梯度 | w 增加则 L 增加 | w 减少 | L 减少 ✅ |

负梯度方向 = 函数值下降最快的方向，这就是**最速下降法**（Steepest Descent）。

---

## 1-5-9 一阶 vs 二阶优化：工业界的现实选择

### 一阶方法的本质

一阶方法只用**一阶导数（梯度）** 来更新参数：

```
w_new = w - lr × ∇L(w)
```

**优点**：
- 计算量小：只需一次前向 + 一次反向，O(n)
- 存储少：只需存梯度向量，O(n)

**缺点**：
- 用线性近似对付非线性曲面，需要多步迭代
- 无法利用曲率信息，不知道步子该迈多大

### 二阶方法的本质

二阶方法利用**二阶导数（Hessian 矩阵）** 来更新参数：

```
w_new = w - H⁻¹ · ∇L(w)
```

其中 `H[i,j] = ∂²L/∂wᵢ∂wⱼ` 是所有二阶偏导数构成的矩阵。

**优点**：
- 对二次函数一步到位（理论最优）
- Hessian 包含曲率信息，步长自然确定，无需学习率

**缺点**：
- Hessian 是 n×n 矩阵，存储 O(n²)
- 计算 Hessian 或其逆是 O(n²) 或更高
- 对于 70B 参数的模型，Hessian 完全无法存储和计算

### 为什么工业界选一阶？

| 维度 | 一阶 | 二阶（Hessian） |
|---|---|---|
| 存储 | O(n) | O(n²) |
| 计算量 | O(n) | O(n²) ~ O(n³) |
| 70B 参数存储 | ~280GB | ~49万亿 GB |
| 可行性 | ✅ | ❌ |

### 工业界的工程 tricks

虽然二阶不可行，但工程 tricks 通过**近似曲率信息**来弥补：

#### 1. 自适应学习率（Adam/RMSProp）

**SGD 的问题**：所有参数用同一个学习率，但不同参数的梯度大小可能差几十倍。

**Adam 解决思路**：让梯度大的时候步子小，梯度小的时候步子大。

---

In [ ]:
# Adam 更新公式
m = β₁·m + (1-β₁)·∇L      # 梯度的一阶矩估计（类似 momentum）
v = β₂·v + (1-β₂)·∇L²     # 梯度平方的 EMA
w = w - lr·m / (√v + ε)

---

| 参数 | 含义 | 初始值 | 固定/可变 |
|------|------|--------|----------|
| `β₁ = 0.9` | m 的衰减率 | 固定 | 超参数，可调 |
| `β₂ = 0.999` | v 的衰减率 | 固定 | 超参数，可调 |
| `lr` | 学习率 | 固定 | 人调，不学习 |
| `ε = 1e-8` | 防止除零 | 固定 | 工程参数 |
| `m` | 梯度 EMA | `0` | 每步更新 |
| `v` | 梯度平方 EMA | `0` | 每步更新 |

**核心机制**：分母 `√v` 自适应调整学习率——梯度大的参数 `√v` 也大，学习率被压小；梯度小的参数 `√v` 也小，学习率被放大。

**与二阶方法的关系**：Newton 法 `w_new = w - H⁻¹·∇L` 用 Hessian 提供最优步长，但 Hessian 存储 O(n²) 不可行。Adam 用 `v` 估计 `E[∇L²]`（Hessian 对角线），用 `m` 估计 `∇L`，本质是**用一阶计算量换二阶效果的部分近似**。这就是 Adam（Adaptive Moment Estimation）名字的由来。

---

In [ ]:
# 直观理解：不同参数自动获得不同学习率
if 梯度大: lr被压小  # 防止震荡
if 梯度小: lr被放大  # 加速收敛

---

RMSProp 和 Adam 思路类似，区别在于 Adam 多了一个 `m`（momentum）来平滑更新方向。

#### 2. 学习率衰减 / 余弦退火

```
lr(t) = lr_max × cos(t/T_max × π)
```

本质：初期大步探索，后期小步收敛。越接近最优点，曲面越接近二次函数，小步长的线性近似更准确。

#### 3. Warmup

```
前N步：lr 从小慢慢增大
之后：正常衰减
```

防止早期曲率估计不稳定时步子太大。

#### 4. 梯度裁剪

---

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

防止梯度爆炸，本质是给步长加了一个安全上界。

### 理论的尴尬现状

| 方法 | 理论证明 | 实际情况 |
|---|---|---|
| 梯度下降（凸） | 严格收敛 | 非凸下不保证 |
| Adam | 有收敛证明 | 假设与实际场景不符 |
| AdamW | 无 | 经验性，工业界大量使用 |
| Warmup | 无 | 经验性，效果稳定 |

**一句话总结**：深度学习优化器是"工程经验驱动，理论严重滞后"的领域。一阶方法是当前大规模训练的唯一可行解，工程 tricks 本质上都是在用近似曲率信息弥补二阶信息的缺失。

---

## 1-5-10 PyTorch autograd 核心 API 速查

### 计算图与梯度追踪

| API | 作用 | 典型用法 |
|---|---|---|
| `requires_grad=True` | 开启当前张量的梯度追踪 | `x = torch.tensor([1.], requires_grad=True)` |
| `x.requires_grad` | 查看是否追踪梯度 | 条件判断 |
| `x.is_leaf` | 是否叶子节点（用户创建） | 调试时检查 |
| `x.grad_fn` | 反向传播函数引用 | 调试：`print(y.grad_fn)` |
| `x.grad` | 存储累积的梯度值 | 反向后：`print(x.grad)` |

### 反向传播

| API | 作用 | 典型用法 |
|---|---|---|
| `loss.backward()` | 标量反向传播 | `loss.backward()` |
| `loss.backward(gradient)` | 非标量反向传播 | `y.backward(grad_y)` |
| `retain_graph=True` | 保留计算图供二次反向 | `loss.backward(retain_graph=True)` |
| `create_graph=True` | 在 grad 中构建计算图 | 用于求高阶导 |

---

In [ ]:
# 求二阶导示例
x = torch.tensor([2.], requires_grad=True)
y = x ** 3
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]  # 二阶导 = 6x = 12

---

### 梯度控制

| API | 作用 | 典型用法 |
|---|---|---|
| `optimizer.zero_grad()` | 清零所有参数的梯度 | 训练循环必备 |
| `x.grad.zero_()` | 清零特定张量梯度 | 原位操作 |
| `with torch.no_grad():` | 禁用梯度计算 | 推理时 |
| `@torch.no_grad()` | 装饰器版本 | 函数定义 |
| `torch.set_grad_enabled(False)` | 动态开关 | 条件分支推理 |
| `x.detach()` | 分离张量，截断计算图 | 需要非梯度操作时 |
| `x.detach_()` | 原地分离 | 少用 |

### Hook 机制

| API | 作用 | 典型用法 |
|---|---|---|
| `register_forward_hook(fn)` | 拦截前向传播 | 特征提取 |
| `register_backward_hook(fn)` | 拦截反向传播 | 梯度检查/修改 |
| `register_hook(fn)` | 给 grad 注册 hook | 打印或修改梯度 |
| `handle.remove()` | 取消注册 | 防止内存泄漏 |

---

In [ ]:
# 给梯度注册 hook（反向传播时触发）
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2
loss = y.sum()

def print_grad(grad):
    print("梯度:", grad)

y.register_hook(print_grad)
loss.backward()  # 打印：梯度: tensor([2., 4., 6.])

---

### 优化器中的梯度

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 完整训练循环
for data, target in dataloader:
    optimizer.zero_grad()        # Step 1: 清梯度
    output = model(data)          # Step 2: 前向（计算图自动构建）
    loss = criterion(output, target)
    loss.backward()              # Step 3: 反向（梯度存到 parameter.grad）
    optimizer.step()             # Step 4: 用梯度更新参数

---

### 梯度检查（Gradient Checking）

用于验证 autograd 计算的梯度是否正确（数值法近似 vs 解析法）：

---

In [ ]:
def gradient_check(model, x, y, eps=1e-5):
    model.eval()
    x.requires_grad = True
    output = model(x)
    loss = criterion(output, y)
    loss.backward()

    # 数值梯度
    for p in model.parameters():
        if p.grad is not None:
            numerical = []
            analytical = p.grad.data.clone()
            for i in range(min(5, p.numel())):  # 只检查前5个
                old = p.data.view(-1)[i].item()
                p.data.view(-1)[i] = old + eps
                loss_plus = criterion(model(x), y).item()
                p.data.view(-1)[i] = old - eps
                loss_minus = criterion(model(x), y).item()
                numerical.append((loss_plus - loss_minus) / (2 * eps))
                p.data.view(-1)[i] = old
            print(f"数值: {numerical[:3]}, 解析: {analytical.view(-1)[:3]}")

---

## 三、`nn.Module` — 模型构建核心

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 继承 `nn.Module` | 必须重写 `__init__` + `forward` | 为什么要继承 |
| `super().__init__()` | 调用父类构造函数 | 不调用会怎样 |
| `named_parameters()` / `parameters()` | 遍历模型参数 | 如何冻结部分层 |
| `state_dict()` / `load_state_dict()` | 模型序列化/加载 | 怎么只加载部分参数 |
| `children()` / `modules()` | 遍历子模块 | 区别是什么 |
| 常见层 | `Linear / Conv2d / BatchNorm / Dropout / LSTM / Embedding` | 参数含义 |
| Winograd | 卷积计算优化算法 | F(m,r) 通用形式 |

**⚠️ 面试必答题：**
- `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？
- `model(img)` 背后发生了什么？（call → forward → hooks）
- `model.train()` vs `model.eval()` 区别？（BN 和 Dropout 的行为差异）

# 三、nn.Module — 模型构建核心

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 继承 `nn.Module` | 必须重写 `__init__` + `forward` | 为什么要继承 |
| `super().__init__()` | 调用父类构造函数 | 不调用会怎样 |
| `named_parameters()` / `parameters()` | 遍历模型参数 | 如何冻结部分层 |
| `state_dict()` / `load_state_dict()` | 模型序列化/加载 | 怎么只加载部分参数 |
| `children()` / `modules()` | 遍历子模块 | 区别是什么 |
| 常见层 | `Linear / Conv2d / BatchNorm / Dropout / LSTM / Embedding` | 参数含义 |

**⚠️ 面试必答题：**
- `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？
- `model(img)` 背后发生了什么？（call → forward → hooks）
- `model.train()` vs `model.eval()` 区别？（BN 和 Dropout 的行为差异）

---

# 3. nn.Module — 模型构建核心

## 3-1 模块定义与参数管理

### 为什么要继承 nn.Module？

`nn.Module` 是 PyTorch 的模型组织框架。继承它不是为了"继承方法"，而是为了获得它提供的**基础设施**：

1. **自动参数收集** — `model.parameters()` 把所有 `nn.Parameter` 收拢，optimizer 一行搞定
2. **设备统一管理** — `model.to('cuda')` 一次把所有参数和数据搬到 GPU
3. **状态追踪** — 前向/反向 hooks、buffer 管理、模型结构序列化

---

In [ ]:
import torch
import torch.nn as nn

# 不继承 nn.Module：纯手工，每个 tensor 手动管理
W = torch.randn(10, 5, requires_grad=True)
b = torch.randn(5, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.1)  # 手动传参

# 继承 nn.Module：自动化的基础设施
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()                          # ← 必须调用
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self, x):
        return x @ self.weight + self.bias

model = Linear(10, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 一行搞定所有参数

---

### super().__init__() 为什么必须调用？

`nn.Module.__init__()` 里注册了若干 PyTorch 内部簿记结构——参数注册表、buffer 注册表、forward hook 容器等。不调用 super，`self.weight` 就不会被当成参数收集，`model.parameters()` 里就没有它，`model.to('cuda')` 也不会移动它。

---

In [ ]:
class BadLinear(nn.Module):
    def __init__(self, in_f, out_f):
        # 忘记调用 super().__init__()
        self.weight = nn.Parameter(torch.randn(in_f, out_f))  # 不会被注册！

model = BadLinear(10, 5)
print("parameters:", list(model.parameters()))  # [] — 空！weight 丢失了
print("weight 是否在 params:", model.weight in model.parameters())  # False

---

### nn.Parameter 是什么？

---

In [ ]:
nn.Parameter(tensor) ≈ requires_grad=True + 自动注册到父 Module

---

| | 普通 `torch.tensor` | `nn.Parameter` |
|--|---|---|
| `requires_grad` | 默认 False | 默认 True |
| 被 `parameters()` 收集 | ❌ | ✅ |
| 能被 optimizer 更新 | ❌ | ✅ |
| 本质 | 就是 tensor | 就是 tensor（外面包了注册逻辑） |

`nn.Parameter` 本质上就是个 `requires_grad=True` 的 tensor，只是外层套了一层"把自己注册到父 Module"的逻辑。没有其他魔法。

### 计算图视角下的 nn.Module

```
nn.Module (Python 对象)
    ├── self.weight (nn.Parameter = requires_grad=True 的 tensor)
    ├── self.bias   (nn.Parameter = requires_grad=True 的 tensor)
    └── forward()
            ↓
        tensor 运算 → 构建计算图（和纯 tensor 完全一样）
```

**关键**：nn.Module 本身不参与计算图，它只是 Python 层的组织壳。真正在计算图里的是 `nn.Parameter`（tensor）。`model(x)` 执行时，实际上是在 tensor 层面做运算，构建计算图。

---

In [ ]:
# 验证：Module 本身不在计算图里
x = torch.tensor([1., 2.], requires_grad=True)
linear = nn.Linear(2, 3)
y = linear(x)  # linear 是个 Python 对象，不是 tensor

print("linear.weight 在图中:", linear.weight.requires_grad)  # True
print("linear.weight.grad_fn:", linear.weight.grad_fn)      # None — 叶子节点！
print("y.grad_fn:", y.grad_fn)                             # <AddmmBackward>

---

### forward 为什么只需写前向？

因为 PyTorch 的 autograd 引擎根据**前向运算的每一步**，自动构建反向传播图。`forward` 里写了什么运算，autograd 就自动生成对应的反向函数（grad_fn）。

`model(x)` 背后发生了什么：

---

In [ ]:
output = model(x)

# 等价于：
output = nn.Module.__call__(model, x)
# 1. model.__call__() 执行（Python 魔法方法）
# 2. 调用 model.forward(x)                    ← 你的代码在这里
# 3. 执行所有注册的 forward_hooks            ← 可选拦截点
# 4. 返回 output

---

`__call__` 是 Python 的语法糖：任何 `obj(args)` 调用，实际是 `obj.__call__(args)`。PyTorch 在 `nn.Module.__call__` 里插入了一些钩子（forward pre-hook、forward post-hook），所以直接写 `forward` 不够——需要经过 `__call__`。这也是为什么 `model(x)` 和 `model.forward(x)` **不完全等价**。

---

## 3-2 遍历与结构（children / modules / named_parameters）

### 四个遍历方法的区别

---

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 8),
    nn.ReLU(),
    nn.Sequential(
        nn.Linear(8, 4),
        nn.ReLU()
    )
)

---

| 方法 | 返回内容 | 层级 |
|------|---------|------|
| `model.modules()` | 所有模块（包括自己、根） | 深度优先遍历（根 → 子 → 子...） |
| `model.children()` | 直接子模块（不包括嵌套更深） | 仅一层 |
| `model.named_parameters()` | 所有参数的 (name, tensor) | 深度遍历 |
| `model.parameters()` | 所有参数的 tensor | 深度遍历 |

---

In [ ]:
print("=== modules() (所有模块，包括嵌套) ===")
for m in model.modules():
    print(f"  {m}")

print("\n=== children() (仅直接子模块) ===")
for m in model.children():
    print(f"  {m}")

print("\n=== named_parameters() ===")
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

---

输出：
```
=== modules() ===
  Sequential(
    (0): Linear(in_features=10, out_features=8)
    (1): ReLU()
    (2): Sequential(
      (0): Linear(in_features=8, out_features=4)
      (1): ReLU()
    )
  )
  Linear(in_features=10, out_features=8)
  ReLU()
  Sequential(...)
  Linear(in_features=8, out_features=4)
  ReLU()

=== children() ===
  Linear(in_features=10, out_features=8)
  ReLU()
  Sequential(...)

=== named_parameters() ===
  0.weight: torch.Size([8, 10])
  0.bias: torch.Size([8])
  2.0.weight: torch.Size([4, 8])
  2.0.bias: torch.Size([4])
```

### 实际应用：冻结部分层

---

In [ ]:
# 冻结所有 children 中名字包含 'bias' 的参数
for name, param in model.named_parameters():
    if 'bias' in name:
        param.requires_grad = False

# 或者用 children 遍历，子模块是顺序存储的
for i, child in enumerate(model.children()):
    if i < 2:  # 冻结前两层
        for param in child.parameters():
            param.requires_grad = False

# 验证
trainable = [p for p in model.parameters() if p.requires_grad]
print("可训练参数:", sum(p.numel() for p in trainable))

---

### 冻结参数在梯度链中的行为

**冻结参数（`requires_grad=False`）不等于截断梯度**。冻结参数的梯度原封不动继续往上传，冻结只是"不记录该参数的梯度"。

---

In [ ]:
import torch

x = torch.tensor([1., 2.], requires_grad=True)
w_frozen = torch.tensor([[1., 0.], [0., 1.]], requires_grad=False)  # 冻结
w_trainable = torch.tensor([[2., 0.], [0., 2.]], requires_grad=True)  # 可训练

y = x @ w_frozen @ w_trainable
loss = y.sum()
loss.backward()

print("x.grad:", x.grad)              # 有梯度（w_frozen 没阻断）
print("w_trainable.grad:", w_trainable.grad)  # 有梯度

---

梯度路径：`loss → w_trainable → w_frozen → x`

w_frozen 被跳过（不记录梯度），但梯度继续传给它上游的 x。

**冻结 vs detach 的本质区别**：

| 操作 | 梯度流过？ | 记录该参数梯度？ |
|------|-----------|----------------|
| `requires_grad=False` | ✅ 畅通，梯度原封不动 | ❌ 不记录 |
| `detach()` | ❌ **截断**，梯度停止 | — |

**冻结参数的典型应用**：预训练模型 backbone 冻住，只 fine-tune 头部。backbone 的权重作为已知的"常量"，只更新 head 的参数——backbone 本身不更新，但梯度仍然流过它。

---

## 3-3 state_dict 与模型保存加载

### 两种保存方式的区别

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 2)

# 方式一：保存整个模型（不推荐）
torch.save(model, '/tmp/model_full.pth')  # 包含类定义，重载依赖类
loaded1 = torch.load('/tmp/model_full.pth')

# 方式二：保存 state_dict（推荐）
torch.save(model.state_dict(), '/tmp/model_sd.pth')  # 只有参数字典
loaded2 = nn.Linear(10, 2)
loaded2.load_state_dict(torch.load('/tmp/model_sd.pth'))  # 需要先实例化

---

**为什么推荐 state_dict**：
- 文件更小（只存参数，不存模型结构）
- 不依赖类定义，迁移性更强
- 断点续训时通常需要同时保存 optimizer 的 state_dict

### 完整断点续训

---

In [ ]:
# 保存
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.234,
}
torch.save(checkpoint, '/tmp/ckpt.pth')

# 加载
ckpt = torch.load('/tmp/ckpt.pth')
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch'] + 1

---

### 部分加载（strict / 非 strict）

---

In [ ]:
# 场景：预训练模型有 100 个参数，但你的模型改了最后层，只有 80 个

# strict=False：忽略不匹配的 key
pretrained = {'layer1.weight': ..., 'layer1.bias': ..., 'layer2.weight': ...}
model = MyModel()  # 只有 layer1
model.load_state_dict(pretrained, strict=False)  # layer2 缺失被忽略

# 精确过滤：只加载部分参数
pretrained = torch.load('/tmp/pretrained.pth')
partial = {k: v for k, v in pretrained.items() if 'fc' not in k}
model = MyModel()
model.load_state_dict(partial, strict=False)

---

---

## 3-4 train / eval 模式与 BatchNorm / Dropout 机制

### Dropout 机制

---

In [ ]:
import torch
import torch.nn as nn

dropout = nn.Dropout(p=0.5)  # 训练时随机丢弃 50%

x = torch.ones(5)
print("train 模式:", dropout(x))   # 训练时有随机性，部分元素变 0
print("eval 模式:", dropout.eval()(x))  # eval 时所有元素保留原值

# 注意：eval 模式需要用 dropout.eval()，而不是 dropout(x) 在 eval 之后调用
dropout.train()   # 切换到训练模式
dropout.eval()    # 切换到评估模式

---

**Dropout 训练 vs 评估的行为差异**：

| 模式 | 行为 | 效果 |
|------|------|------|
| train | 随机置零（p 的概率），剩余元素除以 `(1-p)` 做缩放 | 防止过拟合 |
| eval | 所有元素原样通过 | 确定性输出 |

**缩放的必要性**：`E[dropout(x)] = (1-p) × x/(1-p) = x`，数学期望不变，训练和测试的输出期望一致。

### BatchNorm 机制

---

In [ ]:
import torch
import torch.nn as nn

bn = nn.BatchNorm1d(3)  # 3 个特征

x = torch.randn(4, 3)  # batch=4, features=3
print("train 模式:")
out_train = bn(x)
print("  输出:\n", out_train)
print("  mean per feature:", out_train.mean(dim=0))  # ≈ [0, 0, 0]
print("  var per feature:", out_train.var(dim=0))   # ≈ [1, 1, 1]

print("\neval 模式:")
out_eval = bn.eval()(x)
print("  输出:\n", out_eval)  # 和 train 完全不同！

---

**BatchNorm 训练 vs 评估的行为差异**：

| 模式 | 计算均值/方差 | 使用均值/方差 |
|------|-------------|-------------|
| train | 用当前 batch 的统计量 | 当前 batch 的统计量 |
| eval | 不计算 | 用**全局移动平均**（moving average，训练时累积的） |

---

In [ ]:
# 验证：eval 用的 moving average 是训练时累积的
bn = nn.BatchNorm1d(3)
bn.train()

print("training 模式:")
for i in range(3):
    x = torch.randn(4, 3) * (i + 1)  # 越来越大的方差
    out = bn(x)
    print(f"  batch {i} mean: {out.mean(dim=0)}")

print("\n切换 eval 后（使用 moving average）:")
bn.eval()
x = torch.randn(4, 3)
print("  输出:", bn(x))
print("  running_mean (不更新):", bn.running_mean)
print("  running_var (不更新):", bn.running_var)

---

### model.train() vs model.eval() 实际影响

---

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(0.3)
)

model.train()
print("train 模式 - BatchNorm 用 batch 统计:")
x = torch.randn(8, 10)
print("  BatchNorm running_mean:", model[1].running_mean[:3])

model.eval()
print("\neval 模式 - BatchNorm 用 moving average:")
print("  BatchNorm running_mean:", model[1].running_mean[:3])

---

**总结：哪些层受 train/eval 影响？**

| 层 | train 行为 | eval 行为 |
|---|---|---|
| BatchNorm | 用 batch 统计量 | 用 moving average |
| Dropout | 随机置零 | 全部通过 |
| LayerNorm | 用 batch 统计量 | 用 batch 统计量（不变化） |
| InstanceNorm | 用 batch 统计量 | 用 running average |

---

### 补充：Dropout 的反向传播机制

Dropout 的前向过程是 `out = x * mask / (1-p)`，其中 mask 是随机生成的 0/1 掩码。

**反向传播时，mask 本身不参与梯度计算**（没有梯度），所以：

```
上游梯度 × mask / (1-p)
→ 被 mask=0 置零的位置，梯度 = 0（该神经元这轮不更新）
→ 被 mask=1 保留的位置，梯度正常回传
```

**关键**：Dropout 不阻断梯度流，只是"杀死"被 mask 置零的那些神经元，让它们这轮不更新。下一轮前向时重新生成 mask，神经元可能又活了。

**极端 Bottleneck 场景**：如果网络的窄层只有一个参数连接到前后两部分，该参数被 mask=0 置零时，前后两部分的梯度都会变成 0——这轮训练完全失效。实际网络有冗余路径，所以不严重。

### 补充：Norm 家族深度解析 — CV vs NLP 的本质差异

#### 三种 Norm 的归一化维度对比

| Norm | 归一化维度 | 归一化时参考的样本/位置 | 典型场景 |
|------|-----------|----------------------|---------|
| **BatchNorm** | 每个通道，跨 batch | 同一通道的所有样本同一位置 | 图像分类（CNN） |
| **LayerNorm** | 每个样本，跨所有维度 | 每个样本内部 | Transformer / NLP |
| **InstanceNorm** | 每个样本的每个通道 | 每个样本 × 每通道的空间区域 | 风格迁移 |

#### 核心直觉：在哪个维度做 Norm，就是消除哪个维度的差异

| Norm | 消除的差异 | 保留的差异 |
|------|-----------|-----------|
| **BatchNorm** | 不同样本在同一通道上的绝对水平差异 | 同一样本内部通道间的相对关系 |
| **LayerNorm** | 每个样本内部向量各维度的绝对强度 | 向量在高维空间里的**方向**（语义主要在方向里） |
| **InstanceNorm** | 每个样本每个通道的全局统计量 | 内容结构，去掉风格纹理 |

#### BatchNorm 为什么对图像有效，对 NLP 无效？

**图像**：不同图片在同一个空间位置 (h, w) 具有相同的**视觉语义含义**。所有图片在 (5, 7) 位置都是"某个局部纹理或边缘"，通道 C 在该位置的激活值跨图片具有相似的统计分布——BatchNorm 的假设成立。

**NLP**：第 0 个词和第 0 个词之间没有任何语义对齐。"The" 和 "A" 都出现在句子开头，但含义完全无关。跨样本在同位置做归一化，强行把不同语义的 token 拉成相同分布，破坏了语义信息。

---

In [ ]:
# 两个句子，同一位置 pos=0
# 句子A："The cat sits"     → token[0] = "The"
# 句子B："A dog runs"       → token[0] = "A"
# BatchNorm 强制让两个位置的激活值统计分布相同——灾难性的

---

#### LayerNorm 保留的是什么？

LayerNorm 对每个 token 的向量做归一化，数学上等价于将向量投影到**高维球面上**（固定模长，消除绝对强度）。语义信息主要编码在**方向**里，而不是模长里——所以 LayerNorm 消除模长的同时保留了语义。

#### 验证：NLP 场景下 BatchNorm 的问题

以 NLP 输入 (batch=2, seq_len=3, embed=4) 为例：

---

In [ ]:
# 句子A：[1,2,3,4], [5,6,7,8], [9,10,11,12]
# 句子B：[10,10,10,10], [20,20,20,20], [30,30,30,30]

# LayerNorm 后：句子A token[0] = [-1.34, -0.45, 0.45, 1.34]（递增关系保留）
#              句子B token[0] = [-1.34, -1.34, -1.34, -1.34]（各维度相同，LayerNorm后仍相同）

# BatchNorm 后（embed 维度跨 batch 归一化）：
# 句子A token[0] 和 句子B token[0] → 相同值！
# 两个语义完全不同的句子被强制拉平了

---

**结论**：BatchNorm 对 NLP 是有害的，因为它破坏了不同 token 之间的语义差异。

#### 现代大模型为什么不用 BatchNorm？

1. **分布式训练的 batch 统计量不稳定**：多 GPU 分割 batch 后，各 GPU 看到的统计量差异大
2. **序列位置无结构**：前面论证过
3. **训练/推理不一致（TID，Training Inference Discrepancy）**：NLP 任务中 batch 统计量和 running statistics 差异大，导致 BN 在 NLP 表现差
4. **主流架构 Pre-LN 的归一化位置不适用 BN**

现代大模型全部使用 **LayerNorm** 或 **RMSNorm**（只缩放不偏移，计算更快）。

---

## 3-5 常见网络层（Linear / Conv2d / LSTM / Embedding）

### nn.Linear — 全连接层

---

In [ ]:
import torch
import torch.nn as nn

# in_features: 输入特征维度
# out_features: 输出特征维度
# bias: 是否有偏置（默认 True）
linear = nn.Linear(10, 5)  # input: (..., 10) → output: (..., 5)

x = torch.randn(2, 10)     # batch=2, 每个样本10维
y = linear(x)              # (2, 5)
print(y.shape)             # torch.Size([2, 5])

# 验证：y = x @ W^T + b
manual = x @ linear.weight.T + linear.bias
print("手动计算一致:", torch.allclose(y, manual))  # True

---

**参数数量**：`weight: (5, 10)` + `bias: (5,)` = 55 个参数

### nn.Conv2d — 卷积层

---

In [ ]:
import torch
import torch.nn as nn

# in_channels: 输入通道数（灰度图=1，RGB=3）
# out_channels: 输出通道数（卷积核数量）
# kernel_size: 卷积核大小
# stride: 步长（默认1）
# padding: 填充（默认0，same 填充需要 padding=kernel//2）

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

x = torch.randn(1, 3, 32, 32)  # (batch=1, C=3, H=32, W=32)
y = conv(x)                      # (1, 16, 32, 32)
print(y.shape)

# 验证输出尺寸公式
# H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
h_out = (32 + 2*1 - 3) // 1 + 1
print("计算 H_out:", h_out)  # 32 ✓

---

**参数数量计算**：`kernel × kernel × in_channels × out_channels + out_channels`
= `3 × 3 × 3 × 16 + 16` = `432 + 16` = 448 个参数

### 池化层

---

In [ ]:
# MaxPool2d：取最大值
mp = nn.MaxPool2d(kernel_size=2, stride=2)  # 尺寸减半
# AdaptiveAvgPool2d：输出固定尺寸，自适应核大小
ap = nn.AdaptiveAvgPool2d(output_size=(1, 1))  # 输出 (N, C, 1, 1)
# Global Average Pooling: 常见技巧，用自适应池化实现
gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))  # 把每个通道压成 1 个值

---

### nn.LSTM — 循环层

---

In [ ]:
import torch
import torch.nn as nn

# input_size: 每个时间步的输入特征维度
# hidden_size: 隐藏状态的特征维度
# num_layers: LSTM 层数（堆叠）
# batch_first: True → input shape: (batch, seq, feature)

lstm = nn.LSTM(input_size=10, hidden_size=32, num_layers=2, batch_first=True)

x = torch.randn(4, 8, 10)  # (batch=4, seq_len=8, input_size=10)
h0 = torch.zeros(2, 4, 32)  # (num_layers, batch, hidden_size)
c0 = torch.zeros(2, 4, 32)  # (num_layers, batch, hidden_size)

output, (hn, cn) = lstm(x, (h0, c0))
print("output:", output.shape)    # (4, 8, 32) — 每个时间步的输出
print("hn:", hn.shape)           # (2, 4, 32)  — 最后一个时间步的隐藏状态
print("cn:", cn.shape)           # (2, 4, 32)  — 最后一个时间步的细胞状态

# 如果只想要最后一个时间步的输出（通常用这个）
last_output = output[:, -1, :]  # (4, 32)
print("last:", last_output.shape)

---

### nn.Embedding — 词嵌入层

---

In [ ]:
import torch
import torch.nn as nn

# num_embeddings: 词表大小（最大索引 + 1）
# embedding_dim: 每个词的向量维度
emb = nn.Embedding(num_embeddings=10000, embedding_dim=128)

# 输入：整数索引（LongTensor），范围 [0, num_embeddings)
x = torch.tensor([123, 456, 789])  # batch=3 的句子（每个词是一个索引）
vec = emb(x)                         # (3, 128) — 查表得到三个词的向量
print(vec.shape)                      # torch.Size([3, 128])

# 预训练词向量加载
pretrained = nn.Embedding.from_pretrained(torch.randn(10000, 128))
pretrained.weight.requires_grad = False  # 冻结，只fine-tune顶层

---

#### Embedding 前向传播：数学等价 vs 工程实现

Embedding 在数学上等价于 one-hot 向量与 embedding 矩阵的矩阵乘法，但工程实现是直接内存索引，两者路径不同但结果相同：

| 视角 | 实现 | 路径 |
|------|------|------|
| **数学上** | `one_hot(token_id) @ embedding_table` | 构造稀疏向量 → 矩阵乘 → 得到向量 |
| **工程上** | `embedding_table[token_id]` | 直接地址寻址，无矩阵运算 |

工程上就是 `array[index]` 的直接寻址，和哈希表查表没有本质区别。数学上写成矩阵乘法是为了理论推导方便。

#### Embedding 反向传播：梯度怎么传

核心逻辑：**谁用过我，谁把梯度传给我**。

---

In [ ]:
import torch
import torch.nn as nn

emb = nn.Embedding(num_embeddings=10000, embedding_dim=128)
optimizer = torch.optim.SGD(emb.parameters(), lr=0.01)

# 模拟前向：token_id=42 被使用了两次
input_ids = torch.tensor([42, 7, 42, 99])

# 前向
vec = emb(input_ids)  # shape: (4, 128)
loss = vec.sum()       # 简单 loss

# 反向
loss.backward()

# 验证：token_id=42 被使用了两次，梯度应该累加
print("emb.weight.grad[42]:", emb.weight.grad[42])  # 非零梯度
print("emb.weight.grad[7]:", emb.weight.grad[7])    # 非零梯度
print("emb.weight.grad[99]:", emb.weight.grad[99])  # 非零梯度
print("emb.weight.grad[0]（未使用）:", emb.weight.grad[0])  # 接近零

---

**关键行为**：
- 同一个 token 被多次查询（重复词）→ **梯度累加**
- 没被查过的 token → 梯度为 0，不参与更新
- 计算量正比于 token 数（batch × seq_len），不依赖词表大小

#### 预训练词向量加载与微调策略

实际工程中几乎不会从零训练 embedding，而是加载预训练向量：

---

In [ ]:
import torch
import torch.nn as nn

# 方式一：from_pretrained 加载
pretrained_vectors = torch.randn(10000, 128)  # 实际从 glove/word2vec 加载
emb = nn.Embedding.from_pretrained(pretrained_vectors, freeze=False)
# freeze=False: 训练时继续更新向量（fine-tune）
# freeze=True:  训练时冻结，当静态特征用

---

**四种微调策略**：

| 策略 | 做法 | 适用场景 |
|------|------|---------|
| **Frozen** | `weight.requires_grad = False`，不更新 | 数据量小、领域差异大 |
| **Full Fine-tune** | 全部可训练，正常更新 | 数据量大、领域相近 |
| **Gradual Unfreezing** | 先冻住 → 逐步解冻 | 中等数据量，避免灾难性遗忘 |
| **Adapter** | 冻住主模型，附加小型 MLP 适配器 | 算力有限、保留主模型能力 |

---

In [ ]:
# Gradual Unfreezing 示例：先只训练 head，逐步解冻
model = torchvision.models.resnet18(pretrained=True)

# 阶段1：只训练分类头
for param in model.parameters():
    param.requires_grad = False
model.fc.weight.requires_grad = True

# 训练若干 epoch 后...
# 阶段2：解冻最后两层
for param in model.layer4.parameters():
    param.requires_grad = True

# 阶段3：更多层...
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3  # 解冻后用较小学习率
)

---

---

### Winograd 卷积优化算法

Winograd 算法是深度学习中常用的卷积计算优化技术，核心思想是将卷积中的乘法次数减少。

#### 一维 Winograd F(m, r)：矩阵乘视角

以 F(2, 3) 为例：输入 4 点 `d = [d0, d1, d2, d3]`，滤波器 3 点 `g = [g0, g1, g2]`，输出 2 点：

**直接卷积**（6 次乘）：
```
y0 = d0·g0 + d1·g1 + d2·g2
y1 = d1·g0 + d2·g1 + d3·g2
```

**Winograd 重写形式**：输入组织成 2×3 数据矩阵，滤波器作为 3×1 向量：
```
D = [d0, d1, d2; d1, d2, d3]   (2×3)
y = D · g                        (矩阵乘)
```
这只是一个框架，Winograd 的精妙之处在于：**数据 D 和滤波器 g 都做预变换**，使得主要乘法变成元素乘。

**Winograd 完整公式**：
```
y = A^T · (G·g ⊙ B^T·d)
```

| 步骤 | 操作 | 计算类型 |
|------|------|---------|
| `B^T·d` | 输入变换（4 → 4 点） | 只有加减，无乘 |
| `G·g` | 滤波器变换（3 → 3 点） | 只有加减，无乘 |
| `⊙` | 逐元素乘（4 次） | **只有乘** |
| `A^T` | 输出组合（4 → 2 点） | 只有加减，无乘 |

**总乘法数**：从 6 次降到 **4 次**，代价是十几步加减法。在硬件上乘比加贵得多，所以合算。

#### Winograd F(m, r) 一般形式

Winograd 不是只能 F(2,3)，而是有很多组 (m, r) 可选：

| 形式 | 输出 m | 滤波器 r | 输入点数 | 直接乘法 | Winograd 乘法 |
|------|--------|---------|---------|---------|--------------|
| F(2, 3) | 2 | 3 | 4 | 6 | **4** |
| F(4, 3) | 4 | 3 | 6 | 12 | **6** |
| F(6, 3) | 6 | 3 | 8 | 18 | **8** |
| F(2, 5) | 2 | 5 | 6 | 10 | **6** |

**选大 m 的权衡**：m 越大乘法节省比例越高，但变换矩阵复杂度增加、数值稳定性变差。实际深度学习中选择 F(4,3) 或 F(16,3) 等。

#### 二维 Winograd F(2×2, 3×3)：Khatri-Rao 视角

对于 4×4 输入和 3×3 滤波器，可以分块成 2×3 的 tile，每个 tile 做 2D 卷积：

- 输入 4×4 → 分成重叠的 2×3 块（每个块覆盖一个 2×2 输出区域）
- 每个块是 **2×3 小矩阵**，滤波器是 **3×3 小矩阵**
- 2D 卷积等效为：每个 2×3 块与 3×3 滤波器的矩阵乘

这就是 **Khatri-Rao 乘积**（列对列的逐元素乘）的形式推广。二维 Winograd 的核心洞察：**空间分块 + 滤波器 Khatri-Rao 展开**，使得主要运算变成逐元素乘。

#### 输入不能整除 tile 怎么办

三种处理方式：

**1. 补零（Zero Padding）**：不够的地方补 0，继续用标准 tile，最常用

**2. 剩余处理（Guard / Tail）**：先用标准 tile 覆盖主体，剩下几个点单独用直接卷积（量小，浪费一点乘法无所谓）

**3. 动态选择 tile 大小**：输入长度不是 m 的倍数时，选不规则的最后 tile

#### Winograd 在深度学习中的地位

Winograd 2015-2016 年被工程化后，广泛用于 CNN 推理优化（如 TensorFlow、TensorRT）。但近几年，随着 Tensor Core 等专用矩阵乘硬件的普及，Winograd 在大 tile 场景的优势被削弱——硬件直接做矩阵乘已经足够快。但在**小滤波器（3×3）、中等 batch** 的场景下，Winograd 仍然是重要的优化手段。

---

### 卷积的工程实现：从 for 循环到 im2col 到 cuDNN

#### 为什么 for 循环做卷积极慢

用 for 循环实现单次卷积：

---

In [ ]:
# 最 naive 的实现（假设 stride=1, padding=0）
for oh in range(out_h):
    for ow in range(out_w):
        for kh in range(kernel_h):
            for kw in range(kernel_w):
                val += input[oh+kh, ow+kw] * kernel[kh, kw]
        output[oh, ow] = val

---

四层嵌套 + 大量内存访问 → 极慢，无法利用并行。

#### im2col：把卷积变成矩阵乘法

**核心思想**：把输入的每个滑动窗口"展开"成列，把滤波器展开成行，然后做一次矩阵乘。

```
输入 (H×W) + 滤波器 (K×K)
↓ im2col 展开
矩阵 A (out_h×out_w, K×K)  ×  矩阵 B (K×K, out_channels)
↓                            ↓
每个滑动窗口拉成一列          每个滤波器拉成一行
输出 (out_h×out_w, out_channels)
```

以 4×4 输入、3×3 滤波器、stride=1 为例：
```
输入:
  [d00,d01,d02,d03]
  [d10,d11,d12,d13]
  [d20,d21,d22,d23]
  [d30,d31,d32,d33]

im2col 展开后矩阵 A (4×9):
  [d00,d01,d02,d10,d11,d12,d20,d21,d22]   ← 第1个输出位置的窗口
  [d01,d02,d03,d11,d12,d13,d21,d22,d23]   ← 第2个位置的窗口
  [d10,d11,d12,d20,d21,d22,d30,d31,d32]
  [d11,d12,d13,d21,d22,d23,d31,d32,d33]

滤波器展开成矩阵 B (9×1):
  [k00]
  [k01]
  [k02]
  [k10]
  [k11]
  [k12]
  [k20]
  [k21]
  [k22]

输出 = A @ B  →  矩阵乘!
```

**为什么 im2col 快**：调用高度优化的 BLAS（CPU 上 MKL/OpenBLAS，GPU 上 cuBLAS）矩阵乘，并行度极高。

**内存代价**：im2col 展开后的矩阵 A 有冗余（重叠窗口的数据被重复复制）。但换来的矩阵乘速度提升远大于内存开销。

#### cuDNN：NVIDIA 的卷积底座

cuDNN 是 NVIDIA 提供的深度学习基础库，PyTorch/TensorFlow 等框架的卷积底层都调用它。

**cuDNN 选择算法的方式**：
cuDNN 内部维护多种卷积算法（GEMM-based im2col、Winograd、FFTW 等），每次运行时会对输入 shape 做一次"autotune"——在候选算法上跑一个小算例，选最快的那个缓存下来。

---

In [ ]:
# PyTorch 调用 cuDNN 的示意路径
conv = nn.Conv2d(3, 64, 3)
x = torch.randn(1, 3, 224, 224)

# 实际执行路径：
# 1. PyTorch 解析 conv 参数，构造 cudnnConvolutionDescriptor
# 2. cuDNN 根据 shape 调用 cudnnGetConvolutionForwardAlgorithm (autotune)
# 3. cuDNN 执行选中的算法（im2col + cuBLAS / Winograd / FFTW）
# 4. 结果返回 PyTorch tensor

---

**cuDNN 7 种卷积算法**（按场景选用）：
| 算法 | 适用场景 |
|------|---------|
| GEMM (im2col) | 通用，k×k 大滤波器、large batch |
| Winograd F(2×2, 3×3) | 小滤波器 (3×3)，中等 batch |
| Winograd F(4×4, 3×3) | 较大 tile，batch 大时更明显 |
| FFT | 极大滤波器 (5×5, 7×7)，但内存开销大 |
| Implicit GEMM | 融合了 im2col，不需显式展开，内存更省 |

**cuDNN vs cuBLAS**：cuBLAS 是通用矩阵乘，cuDNN 在其基础上做了 im2col 打包 + filter 打包，然后调 cuBLAS。cuDNN = cuBLAS + im2col + 算法选择 + 反向传播支持。

#### 三种卷积实现的对比

| 实现方式 | 并行度 | 适用场景 | 内存开销 |
|---------|--------|---------|---------|
| for 循环 | 极低 | 教学/验证 | 低 |
| im2col + cuBLAS | 高（矩阵乘） | CPU / 通用 GPU | 中（需展开） |
| cuDNN (autotune) | 最高 | 实际生产 | 可控 |

实际项目中，永远用 cuDNN，PyTorch 默认就调用它，不需要手动指定。

---

### 池化层详解：MaxPool / AvgPool / AdaptiveAvgPool

#### nn.MaxPool2d — 最大池化

---

In [ ]:
import torch.nn as nn

# kernel_size: 池化核大小
# stride: 步长（默认等于 kernel_size）
# padding: 边缘填充
# dilation: 核内空洞间距（膨胀系数）
# ceil_mode: True=向上取整，False=向下取整（默认）

pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1)
x = torch.randn(1, 3, 32, 32)
y = pool(x)  # (1, 3, 16, 16)

---

**前向过程**：在每个 kernel 窗口内取最大值，同时记录最大值位置（indices）。

**反向过程（关键）**：MaxPool 的反向传播**只沿最大值位置传递梯度**，其他位置梯度为 0。这叫 **Max Pooling 的梯度路由**。

---

In [ ]:
# 验证 MaxPool 反向传播
import torch

pool = nn.MaxPool2d(2, stride=2)
x = torch.tensor([[[[1., 2.], [3., 4.]]]], requires_grad=True)
y = pool(x)
loss = y.sum()
loss.backward()

print("x.grad:")  # 只有最大值位置(=4)的梯度为1，其他为0
print(x.grad)     # [[[[0., 0.], [0., 1.]]]]

---

**dilation（膨胀/空洞）参数**：

---

In [ ]:
pool_dilated = nn.MaxPool2d(kernel_size=3, dilation=2)
# 等效感受野 = kernel_size + (kernel_size-1)*(dilation-1) = 3 + 2*1 = 5
# 但实际 kernel 还是 3×3，参数不变，只是采样时跳过了空洞

---

#### nn.AvgPool2d — 平均池化

---

In [ ]:
avg = nn.AvgPool2d(kernel_size=2, stride=2)
# 前向：取 kernel 内均值
# 反向：梯度平均分配给 kernel 内所有位置（= 1/(k*k)）

---

#### Global Average Pooling (GAP)

---

In [ ]:
gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))
x = torch.randn(4, 512, 7, 7)
y = gap(x)  # (4, 512, 1, 1) → squeeze → (4, 512)

---

GAP 将每个通道压缩为 1 个值，完全替代 FC 层。优点：**无参数，不易过拟合**，Inception/ResNet 等现代网络最后常用 GAP。

#### AdaptiveAvgPool2d — 自适应输出尺寸

---

In [ ]:
# 自适应输出到指定尺寸，框架自动计算 kernel_size 和 stride
adaptive = nn.AdaptiveAvgPool2d(output_size=(3, 3))  # 输出 3×3
adaptive = nn.AdaptiveAvgPool2d(output_size=(1, 1))   # 输出 1×1（GAP）

---

无论输入是 7×7 还是 14×14，自适应池化都能输出 3×3，**不需要关心输入尺寸**。

---

### nn.BatchNorm2d — 批归一化深入

#### 训练 vs 推理的统计量差异

---

In [ ]:
bn = nn.BatchNorm2d(num_features=64)

bn.train()   # 训练模式
bn.eval()    # 推理模式

---

| 模式 | 均值/方差来源 | `running_mean/var` |
|------|------------|-------------------|
| train | 当前 batch 实时计算 | **会更新**（指数移动平均） |
| eval | 不计算，用全局统计量 | **不变**，训练时累积的值 |

**running_mean/var 的更新公式**：
```
running_mean = (1 - momentum) * running_mean + momentum * batch_mean
running_var  = (1 - momentum) * running_var  + momentum * batch_var
```

momentum 默认 0.1（PyTorch 中 `momentum=0.1`），意义：**快速追踪近期 batch 统计量变化**。

#### track_running_stats 参数

---

In [ ]:
bn = nn.BatchNorm2d(64, track_running_stats=False)
# 推理时也用 batch 统计量（用于测试集和训练集分布差异大的场景）

---

#### 推理时 BN 的确定性行为

eval 模式下，`BatchNorm` 的行为是**完全确定性**的：

---

In [ ]:
bn.eval()
y1 = bn(x1)  # 无论 x1 是什么，running_mean/var 都固定
y2 = bn(x2)  # y1 和 y2 不可能相等，因为 x 不同 → 但 running_stats 确实不变

---

**关键误解澄清**：eval 模式不是"用 running_mean 代替 batch_mean"，而是：
- **训练阶段**：归一化用 batch 统计量（实时计算）
- **推理阶段**：归一化用 running 统计量（EMA 累积）

两者公式完全一样，只是"均值/方差从哪来"不同。

#### BatchNorm 在什么时候更新 running statistics

---

In [ ]:
bn.train()
bn.eval()  # ← 切换到 eval 后，running_mean/var **完全停止更新**

bn.eval()
bn.train()  # ← 切换回 train 后，继续用 batch 统计量更新 running_mean/var

---

---

### nn.Dropout — 随机失活深入

#### 训练模式的 mask 生成机制

---

In [ ]:
dropout = nn.Dropout(p=0.5)  # p=丢弃概率

x = torch.tensor([1., 2., 3., 4., 5.])
y_train = dropout(x)
# 前向：随机生成 0/1 mask，以 p 概率置 0，剩余 / (1-p)
# 反向：mask=0 的位置梯度=0（不更新），mask=1 的位置正常回传

---

**mask 生成过程（PyTorch 内部）**：

---

In [ ]:
# 伪代码
mask = (torch.rand(x.shape) > p) / (1 - p)  # 注意除以 (1-p)
y = x * mask

---

#### 反向传播：mask 本身不参与梯度计算

Dropout 反向传播的梯度 = 上游梯度 × mask，关键：**mask 在反向时是固定的**（和前向用同一个 mask），mask 本身不产生梯度。

---

In [ ]:
# 验证：Dropout 反向传播
x = torch.tensor([1., 2., 3.], requires_grad=True)
dropout = nn.Dropout(p=0.5)
y = dropout(x)
loss = y.sum()
loss.backward()

# x.grad: 某些位置梯度为 0（被 mask 丢弃），某些位置正常
# x.grad != 0 的位置和前向 mask=1 的位置完全一致

---

#### Inverted Dropout（PyTorch 使用的方式）

训练时做缩放（除以 1-p），推理时不做任何操作：

---

In [ ]:
# Inverted Dropout（PyTorch / TF / 大多数框架）
y = x * mask / (1-p)  # 训练时

# 推理时：所有神经元参与，输出期望 E[y] = x

---

优势：**推理代码和训练代码完全一致**，只需切换 mode。相对于"推理时手动缩放"更简洁。

---

### nn.ConvTranspose2d — 转置卷积（反卷积）

转置卷积不是卷积的逆运算，而是**卷积的梯度运算**（conv transpose = gradient with respect to input）。

---

In [ ]:
conv = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=3, stride=2, padding=1, output_padding=1)
# stride=2: 输出尺寸 = 输入尺寸 × 2（upsample）
# output_padding=1: 消除 stride>1 时的一个单元偏移

---

**用途**：上采样（GAN、U-Net、语义分割）、生成器网络。

**输出尺寸公式**：
```
H_out = (H_in - 1) * stride - 2*padding + dilation*(kernel_size-1) + output_padding + 1
```

---

In [ ]:
# 示例
deconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
x = torch.randn(1, 1, 4, 4)
y = deconv(x)  # (1, 1, 8, 8) — 4×2=8

---

---

### nn.LSTM — 长短期记忆网络深入

#### LSTM 的四个门控机制

---

In [ ]:
import torch.nn as nn

lstm = nn.LSTM(input_size=10, hidden_size=32, num_layers=2, batch_first=True, bidirectional=False)

x = torch.randn(4, 8, 10)  # (batch, seq_len, input_size)
output, (hn, cn) = lstm(x)

---

LSTM 内部有 4 个门控，计算公式：

```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)     # 遗忘门：决定丢弃多少旧信息
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)     # 输入门：决定写入多少新信息
C'_t = tanh(W_C · [h_{t-1}, x_t] + b_C) # 候选记忆：新的候选值
C_t = f_t * C_{t-1} + i_t * C'_t         # 细胞状态：遗忘+输入的组合

o_t = σ(W_o · [h_{t-1}, x_t] + b_o)     # 输出门：决定输出多少
h_t = o_t * tanh(C_t)                    # 隐藏状态：输出门 × tanh(细胞状态)
```

#### 为什么 LSTM 能避免梯度消失

关键在于**细胞状态 C_t 的更新方式**：

```
C_t = f_t * C_{t-1} + i_t * C'_t
```

- 遗忘门 f_t 接近 1 时，梯度可以几乎无损地传回很久以前的时间步
- 加法操作（而非连乘）是 LSTM 避免梯度消失的核心：**∂C_t/∂C_{t-1} = f_t**，不是连乘
- 梯度沿细胞状态路径传递时，"*"操作被"+"替代，梯度传播变成加法而非乘法

#### packed_padded_sequence 与 padding mask

处理变长序列时，需要 pad 后一起 batch，但 pad 的位置不应参与计算：

---

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 变长序列
lengths = [5, 3, 7]  # 各序列实际长度
x_padded = torch.nn.utils.rnn.pad_sequence([...], batch_first=True)

# 打包（按长度降序，自动处理 pad）
packed = pack_padded_sequence(x_padded, lengths, batch_first=True, enforce_sorted=True)
output_packed, (hn, cn) = lstm(packed)

# 解包
output, _ = pad_packed_sequence(output_packed, batch_first=True)
# output[:, 3, :] 在 lengths[0]=5 时，位置3的数据是真实的，位置>3是 pad 的

---

---

### nn.Softmax / LogSoftmax — 归一化指数函数

---

In [ ]:
import torch.nn as nn

sm = nn.Softmax(dim=-1)
log_sm = nn.LogSoftmax(dim=-1)  # 用于 CrossEntropyLoss（数值稳定）

x = torch.tensor([1., 2., 3.])
y = sm(x)  # [e^1, e^2, e^3] / (e^1+e^2+e^3) ≈ [0.09, 0.24, 0.67]
y_log = log_sm(x)  # log([...]) ≈ [-2.41, -1.41, -0.41]

---

**dim 很重要**：在哪个维度做 softmax，就在哪个维度做归一化（和为 1）。

---

In [ ]:
# (batch, seq_len, vocab_size) = (2, 5, 10000)
logits = torch.randn(2, 5, 10000)
sm = nn.Softmax(dim=-1)  # ← 在 vocab_size 维度归一化 → 每词概率分布
attn_weights = sm(logits)  # 每个位置一个概率分布

---

---

### 激活函数对比总结

| 激活函数 | 公式 | 输出范围 | 优点 | 缺点 |
|---------|------|---------|------|------|
| **ReLU** | max(0, x) | [0, +∞) | 计算快、无梯度饱和 | 神经元死亡问题 |
| **LeakyReLU** | x if x>0 else α·x | (-∞, +∞) | 避免神经元死亡 | 多了超参 α |
| **GELU** | x·Φ(x) | (-∞, +∞) | Transformer 最常用，平滑 | 计算略慢 |
| **SiLU / Swish** | x·sigmoid(x) | (-∞, +∞) | 自门控，比 ReLU 更平滑 | 计算慢 |
| **Sigmoid** | 1/(1+e^{-x}) | (0, 1) | 概率输出 | 梯度易饱和、梯度消失 |
| **Tanh** | (e^x - e^{-x})/(e^x + e^{-x}) | (-1, 1) | 零中心 | 梯度易饱和 |

**工程选择**：
- CNN / 通用 → **ReLU**（最快）
- Transformer / Attention → **GELU**（BERT、ViT、GPT 等）
- 需要概率输出 → **Sigmoid**（多标签分类）
- 非线性输出 → **Tanh**（LSTM 门控内部常用）

**GELU vs ReLU**：GELU 是 ReLU 的平滑近似，梯度在负区间也有小幅非零值，不会完全"死亡"。Transformer 时代 GELU 成为默认选择。

---

## 3-6 Sequential 与模块化设计

### nn.Sequential — 两种定义方式

---

In [ ]:
import torch
import torch.nn as nn

# 方式一：直接传入（位置索引，调试不方便）
model = nn.Sequential(
    nn.Linear(10, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)

---

In [ ]:
# 方式二：OrderedDict 命名（推荐，调试友好）
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ('fc1',   nn.Linear(10, 8)),
    ('relu1', nn.ReLU()),
    ('fc2',   nn.Linear(8, 4)),
    ('relu2', nn.ReLU()),
    ('fc3',   nn.Linear(4, 2))
]))

---

### Sequential 内部如何工作

Sequential 本身就是一个 `nn.Module`，`forward` 按顺序执行每一层：

---

In [ ]:
# Sequential 内部 forward 等价于：
def forward(self, x):
    for layer in self._modules.values():
        x = layer(x)
    return x

---

**OrderedDict vs 直接传入的区别**：两者功能完全相同，OrderedDict 只是给每一层起了名字，使得 `named_children()` 和 `state_dict()` 的 key 更清晰。

---

In [ ]:
for name, module in model.named_children():
    print(name, '→', module)

# OrderedDict 输出：
# fc1 → Linear(in=10, out=8)
# relu1 → ReLU()
# fc2 → Linear(in=8, out=4)
# relu2 → ReLU()
# fc3 → Linear(in=4, out=2)

---

### Sequential 的局限：四种必须自定义的场景

Sequential 只能**线性串联**，以下四种情况必须自定义 Module：

**1. 残差连接**

---

In [ ]:
# Sequential 实现不了，必须自定义
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return self.net(x) + x  # ← 跳跃连接

---

**2. 多输入**

---

In [ ]:
# 多输入必须自定义
class TextImageFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_net = nn.Linear(768, 256)
        self.img_net  = nn.Linear(512, 256)

    def forward(self, text, img):
        return self.text_net(text) * self.img_net(img)  # ← 多输入

---

**3. 多输出**

---

In [ ]:
# 多输出必须自定义
class DualOutputNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(512, 10)
        self.embedding  = nn.Linear(512, 128)

    def forward(self, x):
        return {
            'logits': self.classifier(x),
            'features': self.embedding(x)
        }

---

**4. 共享层**

---

In [ ]:
# 同一层在两处使用，必须自定义
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(512, 256)  # ← 两处共享

    def forward(self, x1, x2):
        return self.shared(x1), self.shared(x2)  # ← 同一层走两次

---

### Sequential 适用场景对照表

| 场景 | Sequential | 自定义 Module |
|------|-----------|--------------|
| 线性串联：Conv → BN → ReLU → Pool | ✅ | ✅ |
| 快速原型：几行搭一个 MLP | ✅ | ✅ |
| 残差连接（ResNet Block） | ❌ | ✅ |
| 多分支网络（FPN、Inception、U-Net） | ❌ | ✅ |
| 多输入/多输出（多模态） | ❌ | ✅ |
| 共享层（Siamese Network） | ❌ | ✅ |

### 模型设计的分层思维

```
Input → [Conv → BN → ReLU → Pool] × N → FC → Output
              ↑
         用 nn.Sequential 打包
```

---

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.net(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(3, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.mean(dim=[2, 3])  # Global Average Pooling
        return self.fc(x)

---

### 实际工程选择原则

- **简单任务 / 原型验证** → Sequential 够用，几行代码搞定
- **生产级 / 复杂结构** → 自定义 Module（更清晰可控，可读性好）
- **永远不确定** → 自定义 Module，因为后续改起来容易

---

## 面试常考点

**Q1: `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？**
- PyTorch autograd 根据前向运算自动构建计算图，每种运算对应一个反向函数（grad_fn）
- `model(x)` 实际调用 `nn.Module.__call__(x)` → 执行 forward → 执行 hooks

**Q2: `model(img)` 背后发生了什么？**
- `nn.Module.__call__` 被调用 → 执行 `forward()` → 执行所有注册的 `forward_hooks` → 返回输出
- `__call__` 和直接调 `forward()` 的区别在于 hooks 是否被执行

**Q3: `model.train()` vs `model.eval()` 区别？**
- `train()`：BatchNorm 用当前 batch 的均值/方差，Dropout 随机置零
- `eval()`：BatchNorm 用全局 moving average，Dropout 全部通过
- 两者都只影响特定层的统计行为，不影响其他层

**Q4: 如何冻结部分层？冻结后梯度怎么传？**

---

In [ ]:
for param in model.layer1.parameters():
    param.requires_grad = False
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],  # optimizer 只更新解冻的参数
    lr=0.01
)

---

- 冻结（`requires_grad=False`）：梯度**原封不动**往上继续传，只是该参数本身不记录梯度
- 对比 `detach()`：真正截断梯度流，梯度停止传播
- 冻结不等同于"置零"或"常数"，它仍然参与计算图，只是可学习性被关闭

**Q5: `state_dict()` 保存的是什么？**
- 所有 `nn.Parameter` 的字典，key 是参数名（如 `layer1.weight`），value 是张量
- 不包含模型结构（需要先实例化再 load）

**Q6: `children()` vs `modules()` 区别？**
- `children()`：只返回直接子模块（深度1层）
- `modules()`：返回所有模块，包括嵌套的子子模块（深度遍历）

# 四、torch.optim — 优化器

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| SGD + Momentum | 基础优化器 + 动量加速 | momentum 是什么，为什么能加速 |
| RMSprop / Adagrad | 自适应学习率早期方案 | 解决什么问题 |
| Adam | 自适应学习率 + 偏置校正 | 一阶/二阶矩估计是什么 |
| AdamW | Adam + weight decay 分离 | 为什么比 Adam + L2 好 |
| 学习率调度 | `lr_scheduler` | 常用调度策略及场景 |
| 参数分组 | 不同层不同学习率 | 怎么配 |
| Gradient Clipping | 梯度裁剪 | 防止梯度爆炸 |
| Zero_grad + 状态保存 | 清零梯度 + 保存加载 | 为什么要手动调用 |

**⚠️ 面试必答题：**
- SGD 和 Adam 的区别？各自适用场景？
- AdamW 和 Adam + L2 正则的区别？
- 学习率衰减策略有哪些？
- 为什么梯度要用 `zero_grad()` 清零，不能累加？
- 梯度裁剪具体怎么操作？

---

## 优化器全景对比表

### 7 种优化器对比

| 优化器 | 中文名 | 核心机制 | 优点 | 缺陷 |
|--------|--------|---------|------|------|
| **SGD** | 随机梯度下降 | Δw = lr × g | 稳定，可解释，收敛行为清晰 | lr 固定，调参难，缓坡收敛慢 |
| **Momentum** | 动量法 | v = γ·v + lr·g，Δw = -v | 加速收敛，减少振荡 | 仍需手动调 lr |
| **Nesterov** | Nesterov 动量法 | 先预判位置，再算梯度更新 | 比 Momentum 更稳，提前减速 | 实现稍复杂 |
| **Adagrad** | 自适应梯度算法 | cache += g²，Δw = lr·g/√cache | 每个参数自适应 lr，稀疏特征友好 | cache 只增不减，lr 后期趋 0 |
| **RMSprop** | 均方根传播 | cache = γ·cache + (1-γ)·g² | EMA 平滑，lr 趋向稳定 | 仍需手动调 lr |
| **Adam** | 自适应矩估计 | m/v 双 EMA + 偏置校正，Δw = lr·m̂/√v̂ | 方向+步长同时自适应，几乎不调参 | 显存多两份 state，泛化有时不如 SGD |
| **AdamW** | Adam 权重衰减 | Adam + L2 不进梯度，单独在更新时处理 | 解耦 L2 和 momentum | - |

### 关键概念表

| 概念 | 中文 | 说明 |
|------|------|------|
| EMA | 指数移动平均 | 近远权重 γⁿ 衰减，只保留近期 ~1/(1-γ) 步记忆 |
| 偏置校正 | Bias Correction | EMA 初始化为 0 导致前几步偏低，除以 1-βᵗ 纠正 |
| L2 正则化 | 权重衰减 | 总损失 + (λ/2)Σw²，让参数变小防过拟合 |
| L1 正则化 | Lasso | 总损失 + λΣ|w|，让参数变 0 做特征选择 |
| 有偏估计 | - | 权重之和不等于 1，系统性偏低；除以权重和变为无偏 |

---

## 4-1 SGD 与 Momentum

### SGD（随机梯度下降）

**核心**：每一步独立更新，看当前梯度，不依赖历史。

In [ ]:
Δw = lr × gradient
w = w - Δw

**特点**：
- 稳定，但收敛慢（特别是缓坡）
- 学习率调不好容易震荡

### Momentum（动量）

**核心**：历史梯度的指数加权累积，再用来更新参数。

**公式：**

In [ ]:
momentum = γ × momentum + lr × gradient
w = w - momentum

**展开形式（关键）：**
```
momentum_t = lr × [g_t + γ·g_{t-1} + γ²·g_{t-2} + γ³·g_{t-3} + ...]
```

- γ=0.9 时，10步前的梯度权重只剩约 35%
- 所以 momentum ≈ 过去 ~10 步梯度的加权平均 × lr
- **本质：参数更新量 = lr × 历史梯度的指数加权累积**

**直观例子**（lr=0.1, γ=0.9, momentum=0）：

| Step | 当前梯度 | Momentum | 参数更新 Δw | 累计更新 |
|------|---------|---------|-----------|---------|
| 1 | 3.0 | 0.9×0 + 0.1×3.0 = 0.30 | -0.30 | -0.30 |
| 2 | 2.0 | 0.9×0.30 + 0.1×2.0 = 0.47 | -0.47 | -0.77 |
| 3 | 1.0 | 0.9×0.47 + 0.1×1.0 = 0.52 | -0.52 | -1.29 |
| 4 | 0.5 | 0.9×0.52 + 0.1×0.5 = 0.52 | -0.52 | -1.81 |

**vs 无动量SGD**（同样4步梯度）：累计更新只有 **-0.65**，动量版是 **-1.81**，快了近3倍。

**为什么能加速：**
- 方向一致：速度累积，越走越快
- 方向振荡：正负抵消，自动过滤噪音

**为什么叫"动量"**：物理上像滚球下山，即使坡度变缓，惯性也会推着继续滚。

### Nesterov 动量

**核心**：先按惯性走一步，在那个位置预判梯度，再回来更新。

In [ ]:
preview_w = w - γ × momentum          # 预判位置（先按惯性走）
gradient_preview = 在 preview_w 处算的梯度  # 预判梯度
momentum = γ × momentum + lr × gradient_preview
w = w - momentum

**直观理解**：
- 普通 Momentum：低头冲，撞到墙才减速
- Nesterov：抬头看路，快到墙了提前减速

**代码：**

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, 
                           momentum=0.9, nesterov=True)

**三种方式对比**（lr=0.1, γ=0.9）：

| | 无动量 SGD | 带动量 SGD | Nesterov |
|---|---|---|---|
| 每步更新 | lr × g_t | lr × [g_t + γ·g_{t-1} + ...] | 预判位置算梯度，累积同动量 |
| 速度 | 无 | 有累积 | 有累积 |
| 减速机制 | 无 | 靠当前梯度变小 | **提前**感知梯度变小 |
| 4步累计更新 | -0.65 | -1.81 | 更稳（真实场景优于动量） |

### zero_grad 三种方式

梯度清零有三个入口，语义略有不同：

In [ ]:
optimizer.zero_grad()    # 优化器清零：只清它管理的那些参数的梯度
model.zero_grad()       # 模型清零：递归清所有子模块的梯度
for p in model.parameters():
    p.grad = None       # 手动清零：最直接

**为什么 PyTorch 把 zero_grad 放在 optimizer 上？**
- optimizer 创建时就绑定了要管理的参数
- `optimizer.zero_grad()` = "把管的这批参数的梯度清零"
- 语义自然：谁管谁清

**PyTorch 默认累加梯度**（不是覆盖），方便大 batch：

In [ ]:
loss.backward()      # grad += 梯度
loss.backward()      # grad += 新梯度（累加了！）
optimizer.zero_grad()  # 下一步前必须清零

---

## 4-2 RMSprop / Adagrad

> 这两个算法的核心都是：**每个参数维护自己的 cache，用 cache 来自适应调节学习率**。
> optimizer 内部为每个参数维护一组独立的 cache 值。

### Adagrad

**核心**：历史梯度平方的**累加和**，作为学习率的分母。

In [ ]:
cache_i = cache_i + gradient_i²     # 每个参数单独累积
param_i = param_i - lr × gradient_i / (√cache_i + ε)

**展开形式**：
```
cache_i(t) = g₁² + g₂² + g₃² + ... + g_t²
```

**特点**：
- 稀疏特征友好（梯度小的参数 lr 保持较大）
- cache 只增不减，lr 单调下降
- lr_decay 让 lr 衰减**更快**（叠加在 Adagrad 原有的衰减上），不是更慢

**例子**（lr=0.01, lr_decay=0.01）：

| Step | lr_decay=0 | lr_decay=0.01 |
|------|-----------|--------------|
| 1 | 0.0100 | 0.0099 |
| 100 | 0.0100 | 0.0050 |
| 1000 | 0.0100 | 0.0009 |

**问题**：两层衰减叠加，训练后期 lr 接近 0，无法继续学习。

### RMSprop

**核心**：把 Adagrad 的累加和改成**指数移动平均（EMA）**，让历史梯度的影响自然消退。

In [ ]:
cache_i = γ × cache_i + (1-γ) × gradient_i²
param_i = param_i - lr × gradient_i / (√cache_i + ε)

**展开形式**：
```
cache_i(t) = (1-γ) × [g_t² + γ·g_{t-1}² + γ²·g_{t-2}² + ...]
```

- γ=0.9 时，10 步前的梯度权重只剩约 **3.5%**
- 所以 cache ≈ 最近 ~10 步梯度平方的均值，不是全量累加

**vs Adagrad 对比**：

| | Adagrad | RMSprop |
|---|---|---|
| cache 累积 | 累加和（只增不减） | EMA（γ 衰减权重） |
| lr 变化 | 单调下降 | 趋向稳定值 |
| 需要 lr_decay | 是（额外压制） | 否（EMA 本身是平滑衰减） |
| 适用场景 | 稀疏特征 | 非稀疏 / 深度网络 |

**为什么 RMSprop 不需要 lr_decay**：EMA 天然有"记忆窗口"效果，久远梯度自动被 γ^n 指数衰减压制，不需要额外因子再压 lr。

---

## 4-3 Adam 原理

### Adam（Adaptive Moment Estimation）

**目前最流行的优化器**，结合了 Momentum 和 RMSprop。

## 4-3 Adam 原理

### EMA（指数移动平均）

在讲 Adam 之前，先理解 EMA。

**核心思想**：给近期数据高权重，历史数据权重指数衰减。

In [ ]:
EMA_t = γ × EMA_{t-1} + (1-γ) × x_t

**展开形式**：
```
EMA_t = (1-γ)×[x_t + γ·x_{t-1} + γ²·x_{t-2} + ...]
```

- γ=0.9 时，当前数据权重 0.1，10 步前权重 ≈ 0.035，50 步前 ≈ 0
- 越近的数据权重越大，但更早的数据不会完全消失，而是指数衰减

**为什么叫"指数"**：权重是 γⁿ（指数衰减）。

### 有偏 vs 无偏估计

**无偏估计**：权重之和等于 1，估计的平均值等于真实值。

```
无偏平均 = (x_1 + x_2 + ... + x_t) / t
权重 = [1/t, 1/t, ..., 1/t]，之和 = 1
```

**有偏估计**：权重之和不等于 1，系统性地偏高或偏低。

EMA 的权重之和：
```
(1-γ) + γ(1-γ) + γ²(1-γ) + ... + γ^{t-1}(1-γ)
= (1-γ)(1 + γ + γ² + ... + γ^{t-1})
= (1-γ)(1-γ^t)/(1-γ)
= 1 - γ^t  < 1
```

所以 EMA 是**有偏估计**——权重之和小于 1，导致系统性偏低。

**偏置校正**：除以真实权重之和。
```
EMA校正 = EMA_t / (1 - γ^t)
```
权重之和变成 1，无偏了。

**为什么 EMA 要有偏**：
- 无偏版本需要存储所有历史数据，O(t) 内存
- EMA 用递归形式，O(1) 内存
- 前几步偏差大，但几步后校正项 ≈ 1，几乎不影响
- **有意为之**：用轻微有偏换取工程可行性

### Adam：Momentum + RMSprop

Adam = **一阶矩 EMA（Momentum 方向）** + **二阶矩 EMA（RMSprop 步长）**。

| 组件 | 来源 | EMA对象 | 作用 |
|---|---|---|---|
| **m**（一阶矩） | Momentum | 历史梯度 | 方向：过滤噪音，知道往哪走 |
| **v**（二阶矩） | RMSprop | 历史梯度² | 步长：自适应缩放，知道走多远 |

**一阶矩 m**（管方向）：
- m = β1 × m + (1-β1) × g
- 基于梯度本身，有正负，加算
- sign(m) = 整体运动方向

**二阶矩 v**（管步长）：
- v = β2 × v + (1-β2) × g²
- 基于梯度平方，只有大小，除算
- 控制 lr 的缩放程度

**参数更新**：
```
m_hat = m / (1 - β1^t)    # 偏置校正
v_hat = v / (1 - β2^t)    # 偏置校正
param = param - lr × m_hat / (√v_hat + ε)
```

**偏置校正的推导**：
```
EMA_t = γ·EMA_{t-1} + (1-γ)·x_t
展开 = (1-γ)[x_t + γ·x_{t-1} + γ²·x_{t-2} + ...]
权重之和 = 1 - γ^t

E[EMA_t] = (1-γ^t) × μ  ← 真实均值 μ 的 (1-γ^t) 倍
E[EMA_t / (1-γ^t)] = μ  ← 除以 (1-γ^t) 变成无偏
```

所以校正项 1/(1-β^t) 是从"EMA 是有偏估计"这个事实推导出来的，不是拍脑袋。

### 为什么 Adam 最流行

**Momentum 的缺陷：只管方向，不管步长**
```
m = γ·m + lr·g
```
- 方向靠历史累加，有过滤
- 但步长 = lr × g，lr 设大振荡，设小收敛慢

**RMSprop 的缺陷：只管步长，不管方向**
```
Δw = lr × g / √v
```
- 步长自适应
- 但方向完全跟着当前梯度 g走，没有过滤噪音

**Adam 两者结合**：
```
m = β1·m + (1-β1)·g          # 方向：历史梯度EMA
v = β2·v + (1-β2)·g²          # 步长：历史梯度²EMA
Δw = lr × m / √v              # 方向+步长同时自适应
```

**直观例子**（梯度 = 5, 4, -3）：

| | Momentum | Adam |
|---|---|---|
| 方向 | 累积后正负抵消 | 同左 |
| 步长 | lr × g，固定 | lr × m/√v，自动缩放 |
| 第3步更新 | lr × 0.465 | lr × 约 2.1（更稳定）|

**为什么实践中最优**：
1. 方向+步长同时自适应，几乎不用调参
2. β1=0.9, β2=0.999 基本固定
3. 前几步偏置校正保证正常
4. 收敛快，适合快速实验

**理论缺点（实践中影响小）**：
- 泛化能力有时不如 SGD+Momentum
- 显存比 SGD 多存两份 state（m 和 v）

### Adam 代码

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, 
                          betas=(0.9, 0.999), eps=1e-8)

### Adam 适用场景

| 推荐 | 不推荐 |
|---|---|
| 深度网络（CNN/Transformer） | 简单线性模型 |
| 非稀疏数据 | 需要精确调参的场景 |
| 快速实验阶段 | 对泛化要求极高的任务 |

---

## 4-4 AdamW vs Adam + L2

### L1 / L2 正则化回顾

**正则化的本质**：在损失函数里加一个额外目标，让参数不要太大，防止过拟合。

```
总损失 = 原始损失 + 正则项
```

**L2 正则化**（权重衰减）：`(λ/2) × ||w||²`
- 目标：让所有参数平方和更小
- 梯度贡献：λ × w
- 特点：参数都变小，但都不为 0

**L1 正则化**：`λ × ||w||₁`
- 目标：让尽可能多参数变成 0
- 梯度贡献：λ × sign(w)
- 特点：稀疏化，特征选择

**L2 vs L1 对比**：

| | L2 | L1 |
|---|---|---|
| 公式 | (λ/2)Σw² | λΣ|w| |
| 梯度 | 2λw | λ·sign(w) |
| 特点 | 都变小，趋近0但不为0 | 变成0（特征选择）|
| 大模型为什么用L2 | GPU擅长密集运算，稀疏化收益小；L1梯度不连续不稳定 | |

### Adam + L2 的问题

原始 Adam 论文**没有 L2 正则化**，只是优化 ∇L(w)。后来加了 weight_decay，但放的位置有问题：

**Adam + L2（PyTorch 默认实现）**：

In [ ]:
grad = gradient + weight_decay * param  # L2 混入梯度
m = β1·m + (1-β1)·grad                 # momentum 被污染
v = β2·v + (1-β2)·grad²               # 方差估计也被污染

- L2 的梯度 λ·w 和真实损失梯度混在一起，一起参与 momentum 计算
- momentum 的方向不再只反映真实损失，还被 L2 "往0压"的方向污染
- 方差估计也被 L2 影响

### AdamW（正确实现）

**AdamW**：L2 根本不进梯度，单独在最后参数更新时处理：

In [ ]:
m = β1·m + (1-β1)·gradient  # 干净梯度
v = β2·v + (1-β2)·gradient² # 干净方差
w = w - lr·(m̂/√v̂ + λ·w)    # L2 单独处理，不进 m/v

### 代码实现对比

**Adam（Adam + L2）**：

In [ ]:
# L2 在梯度里混入
if weight_decay > 0:
    grad = grad + weight_decay * p.data
# 然后进入 m 和 v 的计算
exp_avg.mul_(beta1).add_(grad, alpha=1-beta1)
exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1-beta2)

**AdamW（解耦）**：

In [ ]:
# L2 不混入梯度
exp_avg.mul_(beta1).add_(grad, alpha=1-beta1)
exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1-beta2)
# 参数更新后单独处理 L2
p.data.add_(p.data, alpha=-lr * weight_decay)  # w = w - lr·λ·w

### weight_decay 数值：Adam vs AdamW 怎么对应

两者的 λ 不能直接比，因为放置位置不同导致有效强度不同。

AdamW 论文给出精确换算公式：
```
λ_adam ≈ λ_adamw × (1-β1) / √(1-β2)
```

代入 β1=0.9, β2=0.999：
```
(1-0.9) / √(1-0.999) = 0.1 / 0.0316 ≈ 3.16
```

所以：
| AdamW λ | 等价 Adam+L2 λ |
|---------|--------------|
| 0.01 | ≈ 0.003 |
| 0.05 | ≈ 0.016 |

**论文推荐值**：
- Adam + L2：λ = 1e-3 ~ 1e-4
- AdamW：λ = 0.01 ~ 0.05

### 一句话总结

- Adam + L2：L2 进梯度，污染 momentum 和方差估计
- AdamW：L2 不进梯度，单独在参数更新时处理，不污染
- 数值对应：AdamW=0.01 ≈ Adam+L2=0.003（通过公式精确换算）

### 代码

In [ ]:
# Adam + L2（旧方式，不推荐）
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

# AdamW（推荐方式）
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

---

## 4-5 学习率调度（lr_scheduler）

### 为什么需要调度

训练分阶段：
- **前期**：参数离最优解远，需要大学习率快速下降
- **中期**：接近最优解了，大 lr 会振荡
- **后期**：lr 应该变小，精细调整

固定 lr = 0.01：前期快，但后期可能跳过最优点
动态 lr：前期大步快跑，后期小步微调

### 优化器 vs Scheduler

- **优化器**（Adam/SGD）：拿到初始 lr，内部根据梯度自适应缩放每个参数的有效学习率
- **lr_scheduler**：在优化器基础上，再对全局 lr 基准动手脚

两者正交，可以同时用。scheduler 改 `optimizer.param_groups[0]['lr']`，optimizer 拿这个基准做自适应。

### 核心机制

In [ ]:
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

for epoch in range(50):
    train()
    scheduler.step()  # scheduler 改 optimizer 里的 lr 值

scheduler 本质：条件到了，改 `optimizer.param_groups[0]['lr']` 的值，机制简单。

### 七种调度器对比

| 调度器 | 中文名 | 公式 | 触发条件 | 适用场景 | 为什么适合 | 优点 | 缺点 |
|--------|--------|------|---------|---------|---------|------|------|
| **StepLR** | 阶梯衰减 | lr = lr₀ × γ^⌊epoch/s⌋ | 每 N 个 epoch 固定衰减 | 普通训练，不知道曲线怎么走 | 直觉：每隔一段时间降一降；工程：实现简单，step_size 和 gamma 两个超参好调 | 简单稳定 | 需要人工调，机械 |
| **MultiStepLR** | 多阶梯衰减 | lr = lr₀ × γ^milestone_count | 指定 epoch 列表衰减 | 图像分类、目标检测，知道转折点 | 直觉：前期快、中期稳、后期精；工程：CV 训练经验（30/60/80 epoch 降） | 比 StepLR 灵活 | 需要人工指定 milestones |
| **CosineAnnealingLR** | 余弦退火 | lr = η_min + (η_max - η_min) × (1 + cos(π·t/T_max))/2 | 余弦曲线，epoch 线性进行 | 大模型训练，Transformer，BERT/GPT | 直觉：先快后慢，结尾最精细；数学：余弦比阶梯平滑；工程：超参少，效果好，广泛验证 | 平滑不需调参 | 后期 lr 下降慢 |
| **ReduceLROnPlateau** | 自适应衰减 | lr = lr × factor | 验证集指标不降时衰减 | 有验证集、指标明确的训练（翻译/分类/NLP） | 直觉：学不动了再降；工程：不用人工盯 epoch，有监控就能用 | 自适应，无需预知曲线 | 依赖指标质量，波动大时易误触发 |
| **OneCycleLR** | 单周期衰减 | 先升后降：0→lr_max（线性）→lr_min（余弦） | epoch 线性进行，三阶段 | 小 batch 训练、GAN，强化学习、Transformer | 直觉：先探索再收敛；数学：max_lr 让模型跳出局部最优；工程：收敛快，泛化好，只需调 max_lr | 收敛最快，泛化好 | 需调 max_lr |
| **ExponentialLR** | 指数衰减 | lr = lr₀ × γ^epoch | 每个 epoch 衰减 | 几乎不用，实验性调参探索 | 直觉：连续指数衰减；工程：几乎不用，衰减太激进 | 连续平滑 | 衰减太快，难调 |
| **Warmup** | 预热 | lr = lr_target × (step / warmup_steps) | 前 N 个 step 线性上升 | Transformer 训练（标配） | 直觉：先学简单模式再学难的；数学：前几步梯度方向不稳定，小 lr 更安全；工程：前 0.1%~10% 步从小 lr 升到目标值，防止早期震荡 | 稳定训练过程 | 需额外配置 |

### 补充说明

**CosineAnnealing vs StepLR**：余弦不需要提前知道最佳衰减时间点，曲线自动平滑；StepLR 简单但有突变。

**OneCycleLR 为什么收敛快**：max_lr 设置得当可以让模型在前期快速探索大参数空间，后期精细收敛，实践证明比恒定 lr 快 2-5 倍。

**ReduceLROnPlateau 的问题**：指标波动大时会误触发，所以 patience 要设够（5~10 epoch），factor 不能太小（0.5 左右）。

**实际组合**：大模型训练常用 `Warmup + CosineAnnealingLR`——前几个 epoch 预热，后面的 epoch 余弦退火，是 Transformer 的标准配置。

### 代码

In [ ]:
# StepLR
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# CosineAnnealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

# ReduceLROnPlateau
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)
for epoch in range(50):
    val_loss = validate()
    scheduler.step(val_loss)  # 传入指标

# OneCycleLR（Transformer 推荐）
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    epochs=50, steps_per_epoch=len(dataloader)
)

# Warmup + Cosine（手动实现）
warmup_epochs = 5
for epoch in range(50):
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs  # 线性预热
    else:
        progress = (epoch - warmup_epochs) / (50 - warmup_epochs)
        lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))  # 余弦

**面试话术**：
> "常用的是 StepLR（简单）、CosineAnnealing（效果好）和 ReduceLROnPlateau（自动）。Transformer 类任务一般用 Warmup + CosineAnnealingLR，能获得更好收敛。"

---

## 4-6 参数分组（不同层不同学习率）

### 场景

- 预训练模型微调：特征提取层用小学习率，分类头用大学习率
- 不同层用不同 weight_decay

### 为什么分层

拿预训练模型微调举例：

```
模型 = 预训练 backbone（100层，学到的特征很有用）
     + 随机初始化分类头（1层，要从头学）

backbone：lr 太大，已学好的特征会被破坏
分类头：lr 太小，从头学太慢
```

所以：
- backbone（底层特征）：小 lr（1e-4），保护已学知识
- 分类头（顶层）：大 lr（1e-3），快速学习新任务

### 实现

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(), 'lr': 1e-3}
])

### 结合 scheduler

scheduler 衰减是**相对于每个 param_group 的原始 lr**，按比例衰减：

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': backbone, 'lr': 1e-4},   # 原始 lr = 1e-4
    {'params': head, 'lr': 1e-3}        # 原始 lr = 1e-3
])

scheduler = CosineAnnealingLR(optimizer, T_max=50)
# 50 epoch 后：
# backbone lr = 1e-4 × 衰减比例
# head lr = 1e-3 × 衰减比例

scheduler 内部遍历 `optimizer.param_groups`，每个 group 按自己的原始 lr 比例衰减。

### 按参数名筛选

In [ ]:
base_params = [p for n, p in model.named_parameters() if 'head' not in n]
head_params = [p for n, p in model.named_parameters() if 'head' in n]

optimizer = torch.optim.AdamW([
    {'params': base_params, 'lr': 1e-4, 'weight_decay': 0},
    {'params': head_params, 'lr': 1e-3, 'weight_decay': 0.01}
])

### 不同层设不同 lr

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.layer0.parameters(), 'lr': 1e-5},
    {'params': model.layer1.parameters(), 'lr': 5e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-4},
    {'params': model.layer3.parameters(), 'lr': 5e-4},
    {'params': model.head.parameters(), 'lr': 1e-3}
])

### 核心本质

分层设置 → 共用 scheduler → 每个 group 按自己的原始 lr 比例衰减。优化器和 scheduler 算法完全一样，只是初始 lr 不同。

---

## 4-7 Gradient Clipping（梯度裁剪）

### 为什么要裁剪

深层网络（Transformer、LSTM）训练时，梯度逐层累积放大，导致**梯度爆炸**：

```
梯度爆炸：loss.backward() 后，参数梯度变成 NaN 或极大值
→ 参数更新一步跳到无穷大
→ 训练崩溃
```

### 事前防范 vs 事后防范

| 类型 | 方法 | 说明 |
|------|------|------|
| **事前防范** | 残差连接、归一化、合适初始化 | 模型结构设计层面，梯度根本不容易爆 |
| **事后防范** | Gradient Clipping | 训练过程中兜底，已经爆了再裁 |

残差/归一化是"大楼设计时抗震"，Gradient Clipping 是"地震来了再加固"。

### clip_grad_norm_（常用）

按全体参数 L2 范数裁剪，超过阈值等比例压缩：

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

**计算方式**：
```
total_norm = √( ||grad_1||² + ||grad_2||² + ... + ||grad_n||² )
scale = max_norm / total_norm
p.grad = p.grad × scale  # 等比例压缩
```

**例子**：
```
grad = [3.0, 4.0]
||grad|| = √(9+16) = 5.0
total_norm = 5.0, max_norm = 1.0
scale = 1.0 / 5.0 = 0.2
更新后 = [0.6, 0.8]
||grad|| = √(0.36+0.64) = 1.0 ✓
```

### clip_grad_value_（不常用）

每个参数逐元素直接截断到 [-clip_value, clip_value]：

In [ ]:
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)
# grad[i] = min(max(grad[i], -1.0), 1.0)

```
原始：[3.0, -2.5, 0.3, 0.8]
裁剪后：[1.0, -1.0,  0.3, 0.8]
```

**为什么不常用**：
- 直接截断会改变梯度方向
- 离群值（spike）会导致整体缩放剧烈波动
- clip_grad_norm_ 是等比例缩放，保留方向信息

### 执行时机

**每 batch 执行一次**，在 backward() 之后、optimizer.step() 之前：

In [ ]:
for batch in dataloader:
    optimizer.zero_grad()
    outputs = model(batch)
    loss = loss_fn(...)
    loss.backward()                              # 算梯度
    clip_grad_norm_(model.parameters(), 1.0)   # 每 batch 裁剪
    optimizer.step()                            # 更新参数

### Batch 级别 vs Epoch 级别操作

| 操作 | 执行时机 | 说明 |
|------|---------|------|
| optimizer.zero_grad() | 每 batch | |
| forward + loss | 每 batch | |
| loss.backward() | 每 batch | |
| clip_grad_norm_() | 每 batch | |
| optimizer.step() | 每 batch | |
| scheduler.step() | 每 batch 或 每 epoch | 取决于调度器类型 |
| validate() | 每 epoch | 验证集评估 |
| save_checkpoint() | 每 epoch | |

不同调度器的 step() 时机：

| 调度器 | 调用时机 |
|--------|---------|
| OneCycleLR | 每 batch |
| StepLR / CosineAnnealingLR / ReduceLROnPlateau | 每 epoch |

### 代码

In [ ]:
for inputs, targets in dataloader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = loss_fn(outputs, targets)
    loss.backward()
    
    # 梯度裁剪：每 batch 执行，超过阈值等比例压缩
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()

# epoch 结束时
scheduler.step()     # 或 scheduler.step(val_loss)

**面试话术**：
> "梯度裁剪防止梯度爆炸，尤其是 RNN/Transformer。我在实际项目中一般用 clip_grad_norm_，阈值设为 1.0 或 5.0，它按 L2 范数等比例压缩，保留梯度方向信息。"

---

## 4-8 优化器状态保存

### 为什么需要保存 optimizer state

optimizer 内部为每个参数维护 state，比如：
- **SGD with Momentum**：动量 m
- **Adam**：一阶矩 m、二阶矩 v

这些 state 包含了训练过程的"历史信息"，断点续训时不保存 optimizer state，直接从 epoch 50 继续会导致训练行为不一致（相当于从头初始化了 momentum）。

### 保存与加载

In [ ]:
# 保存
checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'epoch': 50,
    'lr_scheduler': scheduler.state_dict()  # scheduler 也可以保存
}
torch.save(checkpoint, 'checkpoint.pth')

# 加载
checkpoint = torch.load('checkpoint.pth')
model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])
epoch = checkpoint['epoch'] + 1  # 从下一个 epoch 继续
scheduler.load_state_dict(checkpoint['lr_scheduler'])

### 注意

- 不同 optimizer 的 state 结构不同，Adam 的 state_dict 不能加载到 SGD
- optimizer state 也是 fp32，显存消耗和模型参数相当（Adam 需要存 m 和 v 两份）

### set_to_none=True（更高效的清零方式）

In [ ]:
optimizer.zero_grad(set_to_none=True)
# grad 设为 None 而非 0，内存更省，效果相同

---

## 面试汇总

### Q1: SGD 和 Adam 的区别？

| | SGD | Adam |
|---|---|---|
| 学习率 | 固定（需手动调） | 自适应 |
| 收敛速度 | 慢 | 快 |
| 调参难度 | 高 | 低 |
| 适用场景 | 简单模型、精确调参 | 深度网络 |

### Q2: AdamW 和 Adam + L2 的区别？

- Adam + L2：L2 正则化进入梯度，影响动量估计
- AdamW：weight_decay 在参数更新时直接减，独立于动量
- 推荐用 AdamW

### Q3: 常用学习率衰减策略？

- StepLR：每 N 个 epoch 固定衰减
- CosineAnnealing：余弦曲线
- ReduceLROnPlateau：验证集不降时自动降
- OneCycleLR：先升后降（Transformer 推荐）

### Q4: 为什么 zero_grad() 必须手动调用？

- PyTorch 默认梯度累加（支持大 batch）
- 每次 backward 会累加到 .grad，所以要清零

### Q5: 梯度裁剪怎么做？

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

## 代码练习

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. 准备数据
X = torch.randn(1000, 10)
y = torch.randn(1000, 1)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32)

# 2. 模型
model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 1))

# 3. 优化器：AdamW + 参数分组
optimizer = optim.AdamW([
    {'params': model[0]..parameters(), 'lr': 1e-3},  # 卷积层
    {'params': model[2].parameters(), 'lr': 1e-4}     # 输出层
], weight_decay=0.01)

# 4. 学习率调度
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 5. 训练循环
for epoch in range(10):
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad(set_to_none=True)
        output = model(batch_x)
        loss = nn.MSELoss()(output, batch_y)
        loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
    scheduler.step()
    print(f"Epoch {epoch}: loss={loss.item():.4f}, lr={optimizer.param_groups[0]['lr']:.6f}")

---

## 参考资料

- PyTorch 官方文档: https://pytorch.org/docs/stable/optim.html
- 论文: Adam: A Method for Stochastic Optimization (2014)
- 论文: Decoupled Weight Decay Regularization (2019) — AdamW

# 五、DataLoader — 数据加载

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| Dataset | 自定义数据抽象，必须实现 __getitem__ + __len__ | 如何自己实现 |
| DataLoader | batch / shuffle / num_workers | 各参数含义 |
| Sampler | Sequential / Random / WeightedRandom | 采样策略是什么 |
| collate_fn | 自定义 batch 拼接 | 什么时候需要重写 |
| pin_memory | 加速 GPU 传输 | 什么原理 |
| drop_last | 丢弃不完整 batch | |
| prefetch_factor + persistent_workers | 多进程预取与保活 | |
| torchvision | 图像领域数据集（MNIST / CIFAR / ImageNet） | |

**⚠️ 面试必答题：**
- DataLoader 的 shuffle 是在哪个层面做的？
- num_workers 设置过大的副作用是什么？
- Sampler 和 shuffle 有什么区别？
- collate_fn 什么时候需要自定义？

---

## 5-1 Dataset

### 核心接口

Dataset 必须实现两个方法：

In [ ]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data_path):
        self.data = load(data_path)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

- `__len__`：返回数据集大小
- `__getitem__`：根据索引返回单个数据

PyTorch 会自动调用这两个方法，DataLoader 不需要知道具体实现。

### TensorDataset（快捷封装）

如果数据已经是张量，不需要自定义：

In [ ]:
from torch.utils.data import TensorDataset

X = torch.randn(1000, 10)
y = torch.randint(0, 2, (1000,))
dataset = TensorDataset(X, y)

### Map-style vs Iterable-style

- **Map-style**（最常用）：实现 `__getitem__`，支持随机索引访问。数据一次性加载到内存，适合能装进内存的数据。
- **Iterable-style**：实现 `__iter__`，支持流式遍历（如从数据库/网络读取数据）。数据按需读取，不撑爆内存。

### Iterable-style 详解（流式数据）

**为什么需要流式？**  
当数据量极大（如1000万条）无法一次装进内存时，需要"随用随取、用多少取多少"。

**yield 的本质**：不是"暂停"，是"冻结"。函数执行到 yield 时，把值吐出去后卡住，下一次 `next()` 从卡住处继续。

In [ ]:
def gen():
    print("A")
    yield 1          # 执行到这里，冻结，吐出 1
    print("B")
    yield 2          # 从这里继续，冻结，吐出 2
    print("C")       # 函数结束，抛 StopIteration

In [ ]:
g = gen()
next(g)  # A + 返回 1
next(g)  # B + 返回 2
next(g)  # C + 抛 StopIteration

**流式数据的自动续接机制**（核心！）：cursor 耗尽后通过异常捕获重新查下一批：

In [ ]:
def __iter__(self):
    offset = 0
    FETCH_SIZE = 100000   # 每批从数据库读多少
    TOTAL_LIMIT = 10000000  # 总数据量

    while True:                              # 永真循环，不退出
        cursor = db.execute(
            f"SELECT * FROM logs LIMIT {FETCH_SIZE} OFFSET {offset}"
        )
        try:
            while True:
                row = next(cursor)           # 逐行取
                yield row                    # 有数据，正常吐出
        except StopIteration:                # cursor 耗尽了，抛异常
            offset += FETCH_SIZE             # offset 跳到下一批
            if offset >= TOTAL_LIMIT:
                return                       # 取够了，epoch 结束
            # 没取够，回到外层 while True，重新建 cursor 继续吐

**生命周期**：yield 出一条 → DataLoader 打包成 batch → 用了 → 再 next() 取下一条 → cursor 耗尽抛异常 → 捕获后重新查下一批 → 继续 yield

**为什么叫"流式"**：数据不是一次性全量加载到内存，而是边流边用。像水流一样，随用随取，不囤积。

**IterableDataset 适合场景**：日志分析、视频流、数据库大表查询结果。Map-style 适合数据能一次性装进内存的场景。

---

## 5-2 DataLoader 核心参数

### 基本用法

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,        # 每个 epoch 是否打乱
    num_workers=4,      # 多少个子进程加载数据
    pin_memory=True,    # 是否锁页内存
    drop_last=False,    # 是否丢弃不完整 batch
    collate_fn=None,    # 自定义 batch 拼接
)

### 逐个参数详解

| 参数 | 类型 | 默认值 | 说明 |
|------|------|-------|------|
| batch_size | int | 1 | 每个 batch 多少样本 |
| shuffle | bool | False | 每个 epoch 是否打乱（仅当 sampler=None 时生效） |
| sampler | Sampler | None | 采样策略（与 shuffle 互斥） |
| num_workers | int | 0 | 子进程数，0=主进程加载 |
| pin_memory | bool | False | 锁页内存，加速 GPU 传输 |
| drop_last | bool | False | 丢弃最后一个不完整 batch |
| collate_fn | callable | None | 自定义 batch 拼接 |
| prefetch_factor | int | 2 | 每个 worker 预取多少 batch |
| persistent_workers | bool | False | epoch 间保持 worker 不退出 |

### 完整参数对照表（默认值 / 推荐值 / 利弊）

| 参数 | 默认值 | 默认原因 | 推荐值 | 优点 | 缺点 / 注意 |
|------|--------|----------|--------|------|-------------|
| batch_size | 1 | 最小单元，不影响功能 | 16/32/64（视GPU显存） | 大batch训练稳定，梯度估计准 | 大batch显存压力大，小batch梯度噪声大 |
| shuffle | False | 不强制打乱，用户自己决定 | 训练：True，验证：False | True时每个epoch数据顺序不同，避免过拟合 | 验证集不需要，打乱反而影响可复现性 |
| sampler | None | 配合shuffle使用，不冲突 | 自定义WeightedRandomSampler处理不平衡 | 可控采样策略 | 与shuffle互斥，设置sampler时shuffle必须False |
| num_workers | 0 | 避免多进程开销，简化调试 | 有预处理时：2~8；纯内存索引：0 | 并行加速IO/预处理，GPU不再空等 | 内存占用高（每worker复制一份数据），进程创建开销，macOS有limitation |
| pin_memory | False | 单CPU训练不需要，节省锁页内存 | GPU训练时：True | DMA直传省去中间拷贝，CPU→GPU加速明显 | 锁页内存占用系统可用内存，过多可能导致OOM |
| drop_last | False | 不丢失数据，最大化利用 | 训练时常用True，验证可用False | batch大小一致，BatchNorm统计稳定 | 少量数据被丢弃（最后一个不完整batch） |
| collate_fn | None | 大多数场景直接stack | 变长序列/嵌套结构：自定义 | 灵活处理各种数据格式 | 不需要时别乱设，会覆盖默认stack行为 |
| prefetch_factor | 2 | 平衡预取与内存 | IO慢/预处理重：增大到4~8；高速存储：默认或1 | 队列始终有货，GPU不等待 | 增大则内存占用上升（num_workers × prefetch_factor × batch_size）|
| persistent_workers | False | 避免占用额外内存 | 多epoch训练：True；单epoch或调试：False | 避免每epoch重新创建worker进程，开销小 | epoch间worker一直存活，内存占用持续 |

> **注意**：prefetch_factor 仅在 num_workers > 0 时生效。persistent_workers 仅在 num_workers > 0 时生效。

### shuffle 的本质

shuffle=True 时，DataLoader 内部使用 `RandomSampler`，本质是在 **样本层面** 打乱索引顺序，而不是打乱 batch。

```
原始数据：[0, 1, 2, 3, 4, 5, 6, 7]
打乱索引：[3, 7, 1, 5, 0, 6, 2, 4]
按 batch 取：[[3,7,1,5], [0,6,2,4]]
```

shuffle 是在 **Sampler 层面** 做的，不是 DataLoader 本身。

**shuffle 与 IterableDataset 不兼容**：shuffle=True 会直接抛 `TypeError: shuffle option is not supported with IterableDataset`。因为流式数据没有 `__len__`，无法构建随机索引。流式数据的"打乱"需在数据库查询层面用 `ORDER BY RANDOM()` 实现。

### num_workers 机制详解

**工作流程（Map-style 数据）**：

```
主进程构建 sampler，fork 出 N 个 worker
每个 worker 复制 sampler 状态，按不同步长取索引：
  Worker1: indices[0::N]
  Worker2: indices[1::N]
  Worker3: indices[2::N]
  ...

各 worker 调用 dataset[idx] 取单样本 → 放进共享队列
主进程：从队列取样本，凑满 batch_size → 打包 → 传给 GPU
```

**为什么 num_workers=0 也够用（但有前提）**：如果 `__getitem__` 只是纯内存索引（如 TensorDataset），几乎没有开销，单进程足够。多进程反而增加进程通信开销。

**num_workers 有意义的前提**：`__getitem__` 里有 IO 或 CPU 密集型操作（读文件、数据增强、tokenization等）。

**IterableDataset + num_workers > 0 的重复问题**：每个 worker 都会重新执行 `__iter__()`，导致数据重复。必须通过 `worker_init_fn` 为每个 worker 分配不同的数据分片。

### pin_memory 机制详解

**普通内存（pinned=False）的传输路径**：

```
数据在普通内存页 → 可能被换出到磁盘
                    ↓
              先换回内存（若被换出）
                    ↓
              拷贝到临时锁页缓冲区
                    ↓
              从缓冲区传到 GPU
```

**锁页内存（pin_memory=True）的传输路径**：

```
锁页内存 → 永远不被换出 → DMA 直传 GPU（无需 CPU 介入）
```

**为什么能加速**：省去了"换页回内存 + 中间缓冲区拷贝"两步。锁页内存通过 DMA（直接内存访问）绕过 CPU 直接传到 GPU。

**锁页大小不需要手动设置**：PyTorch 按需分配，用到时临时锁、传完即释放（或被缓存复用）。Linux 锁页内存总量有限（默认≈RAM的一半），一般训练场景不会触达。

### prefetch_factor 与 persistent_workers

**prefetch_factor 控制的是什么**：不是 worker 每次取多少样本，而是队列里最多囤多少个 batch 的样本。

```
num_workers=4, prefetch_factor=2
→ 每个 worker 预取 2 个 batch 的样本
→ 队列里最多有 4×2=8 个 batch 在等待主进程
→ GPU 始终有货可用，主进程不等待
```

**persistent_workers 的作用**：epoch 结束后 worker 不退出，保持预加载状态。

In [ ]:
# 无 persistent_workers（默认）
for epoch in range(10):
    # 每个 epoch 重新创建 4 个 worker → 进程创建开销 × 10

# 有 persistent_workers=True
# worker 创建一次，10 个 epoch 复用

**典型训练配置推荐**：

In [ ]:
DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,              # 训练用
    num_workers=4,             # 有预处理时
    pin_memory=True,           # GPU 训练时
    drop_last=True,            # 训练时常用
    persistent_workers=True,    # 多 epoch 训练时
)

---

## 5-3 Sampler（采样策略）

Sampler 决定**按什么顺序遍历数据集**。

### 内置 Sampler

| Sampler | 说明 | 场景 |
|---------|------|------|
| SequentialSampler | 按顺序遍历 | 验证/测试集 |
| RandomSampler | 随机遍历 | 训练集（配合 shuffle=True）|
| WeightedRandomSampler | 按权重采样 | 不平衡数据集 |
| DistributedSampler | 分布式训练分片 | 多卡训练 |

### RandomSampler

In [ ]:
from torch.utils.data import RandomSampler

sampler = RandomSampler(dataset, replacement=False, num_samples=1000)
# replacement=False: 每个样本只能被选一次
# num_samples: 采样多少个（可小于数据集大小）

### WeightedRandomSampler（不平衡数据集）

给每个类别设置权重，让模型看到更多少数类：

In [ ]:
labels = [0, 0, 0, 0, 1, 1]  # 0多，1少

# 计算每个样本的权重
class_counts = [4, 2]  # 类别0有4个，类别1有2个
weights = [1.0/class_counts[y] for y in labels]
# weights = [0.25, 0.25, 0.25, 0.25, 0.5, 0.5]
# 类别1的样本权重更高，被采样概率更大

sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

### Sampler vs shuffle

| | Sampler | shuffle |
|---|---|---|
| 作用层面 | 样本遍历顺序 | DataLoader 参数 |
| 灵活性 | 高（可自定义权重/分布）| 低（只有 True/False）|
| 互斥 | 和 shuffle 互斥 | 和 Sampler 互斥 |

**shuffle=True 本质**：DataLoader 内部自动创建 RandomSampler。

### 代码中使用 Sampler

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, WeightedRandomSampler

# 方式1：直接设 shuffle=True（最常用）
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
# 内部自动创建 RandomSampler，不需要碰 Sampler

# 方式2：手动指定 Sampler（不平衡采样 / 分布式训练时才需要）
sampler = WeightedRandomSampler(weights, num_samples=len(weights))
dataloader = DataLoader(dataset, sampler=sampler)  # 不再设 shuffle

注意：`sampler` 和 `shuffle` 互斥，设了 `sampler` 就不能设 `shuffle=True`。

### DistributedSampler 详解

**解决的问题**：多卡训练时，每张卡只跑 1/N 的数据，4张卡合起来覆盖全量数据。

**跳步分区**：

In [ ]:
# 10000条数据，4张卡，rank=0 的卡
indices = list(range(len(dataset)))          # [0, 1, 2, ..., 9999]
indices = indices[rank::num_replicas]       # rank=0 → [0, 4, 8, 12, ...]

| 卡 | rank | 分到的索引 |
|----|------|-----------|
| 卡0 | 0 | 0, 4, 8, 12, ... |
| 卡1 | 1 | 1, 5, 9, 13, ... |
| 卡2 | 2 | 2, 6, 10, 14, ... |
| 卡3 | 3 | 3, 7, 11, 15, ... |

**epoch 间随机打乱（必须调用 set_epoch）**：

In [ ]:
sampler = DistributedSampler(dataset, num_replicas=4, rank=0)

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)  # 驱动随机种子变化，改变打乱方式
    dataloader = DataLoader(dataset, sampler=sampler)
    for batch in dataloader:
        train(batch)

如果不调用 `set_epoch`，每张卡每轮都拿相同的索引顺序，模型会过拟合到"第0卡总拿0,4,8索引"的模式。

**DistributedSampler 本质**：核心就是跳步分区 + epoch 随机种子管理，padding 处理不整除的情况。

---

## 5-4 collate_fn（自定义 batch 拼接）

### 默认行为

DataLoader 把 `dataset[i]` 的返回值打包成一个 batch：

In [ ]:
# dataset[i] 返回 (image, label)
batch = [dataset[0], dataset[1], ..., dataset[31]]
# 等价于：zip([imgs], [labels]) 后 stack

如果所有数据形状相同（如同尺寸图片），默认行为没问题。

### 需要自定义的场景

**场景1：变长序列（如 NLP 的句子，长度不同）**

In [ ]:
def collate_fn(batch):
    # batch = [(seq1, label1), (seq2, label2), ...]
    texts = [item[0] for item in batch]    # 变长列表
    labels = [item[1] for item in batch]
    
    # Padding：把不同长度的序列 padding 到一样长
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    
    return padded_texts, torch.tensor(labels)

dataloader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn)

**场景2：嵌套结构（如检测任务一个样本有多个框）**

In [ ]:
def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]  # 每个样本框数不同，不能 stack
    return images, targets

### 默认 collate_fn 内部逻辑

PyTorch 默认用的是 `default_collate`，按数据类型自动处理：

In [ ]:
def default_collate(batch):
    if all(isinstance(x, torch.Tensor) for x in batch):
        return torch.stack(batch)                # 张量 → stack
    elif all(isinstance(x, np.ndarray) for x in batch):
        return torch.tensor(np.stack(batch))     # numpy → 转tensor后stack
    elif all(isinstance(x, (int, float)) for x in batch):
        return torch.tensor(batch)               # 数字 → 转tensor
    elif all(isinstance(x, str) for x in batch):
        return batch                             # 字符串 → 保持list
    else:
        return batch                             # 其他 → 保持list

所以默认 collate_fn 不是简单 stack，而是**按类型分类处理**。

### collate_fn 里不能做数据增强

数据增强是 CPU 密集操作，放在 collate_fn 里会导致问题：

```
Worker1: __getitem__(0) → 数据增强 → yield
Worker2: __getitem__(1) → 数据增强 → yield
...
        ↓
  主进程收集样本
        ↓
  collate_fn() ← 在主进程里执行！
        ↓
  collate_fn 里的增强 → 主进程被阻塞 → GPU空等
```

- collate_fn 在主进程执行，是**串行**的，增强期间 GPU 空闲
- worker 送完样本后阻塞在"等主进程收"，**并行优势浪费**
- 增强结果不可复用，每批都要重算

**正确做法**：数据增强放在 `__getitem__` 里（worker 并行执行），collate_fn 只做组装：

```
Worker1: __getitem__(0) → 数据增强 → yield  ← 并行算
Worker2: __getitem__(1) → 数据增强 → yield  ← 并行算
        ↓
  主进程收集样本
        ↓
  collate_fn()  ← 只做组装，几乎零开销
        ↓
  GPU训练 ← worker 同时已经在算下一批
```

**结论**：collate_fn 应保持轻量，数据增强必须前置到 Dataset 的 `__getitem__` 里。

---

## 5-5 pin_memory（加速 GPU 传输）

### 原理

| | 普通内存 | pin_memory |
|---|---|---|
| 位置 | 可换出到磁盘 | 锁页内存，不会换出 |
| CPU→GPU 传输 | 先拷贝到临时锁页内存，再传 GPU | 直接传，无需中间拷贝 |
| 速度 | 慢 | 快（尤其大 batch）|

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, pin_memory=True)

### 什么时候用

- 训练在 GPU 上时开启
- 单 CPU 训练（不用 GPU）时不需要
- 结合 `torch.cuda.FloatTensor` 使用效果最佳

### num_workers 的副作用

In [ ]:
dataloader = DataLoader(dataset, num_workers=8)

- num_workers 越大，数据加载越快（并行）
- 但内存占用越高（每个 worker 独立拷贝数据）
- num_workers 过多可能导致：
  - 内存爆炸
  - 进程创建开销大于并行收益
  - 适合的值：2~8，通常 4

---

## 5-6 drop_last / prefetch_factor / persistent_workers

### drop_last（丢弃不完整 batch）

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, drop_last=True)

最后一个 batch 样本数不足 32，会被丢弃。

**为什么需要**：
- batch 大小不同会导致 BN 层统计不稳定
- 某些模型对 batch 形状有要求

### prefetch_factor（预取队列长度）

In [ ]:
dataloader = DataLoader(dataset, num_workers=4, prefetch_factor=2)

每个 worker 预取 2 个 batch 到队列。

总预取数 = num_workers × prefetch_factor

**什么时候调大**：
- CPU 处理快，GPU 等待时 → 增大预取
- 网络/存储 IO 慢 → 增大预取

### persistent_workers（保持 worker 不退出）

In [ ]:
dataloader = DataLoader(dataset, num_workers=4, persistent_workers=True)

epoch 结束后，worker 不退出，保持预加载数据的状态。

**适合场景**：多 epoch 训练，减少 worker 重启开销。

---

## 5-7 torchvision 内置数据集

### 常用数据集

In [ ]:
from torchvision import datasets, transforms

# MNIST（手写数字，0~9，28×28灰度图）
datasets.MNIST(root='./data', train=True, download=True,
               transform=transforms.ToTensor())
# train=True → 6万训练集；train=False → 1万测试集
# download=True 仅在数据不存在时下载，已有缓存则直接使用

# CIFAR-10（10类彩色图：飞机/汽车/鸟/猫/鹿/狗/青蛙/马/船/卡车，32×32）
datasets.CIFAR10(root='./data', train=True, download=True,
                 transform=transforms.Compose([
                     transforms.RandomCrop(32, padding=4),
                     transforms.ToTensor(),
                 ]))

# ImageNet（1000类，需申请下载权限，图片尺寸不固定）
datasets.ImageNet(root='./data', split='train')

### transform 传进 Dataset，不是 DataLoader

`transform` 在 Dataset 的 `__getitem__` 里应用，DataLoader 不直接管 transform：

In [ ]:
# transform 传进 Dataset
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 数据增强
    transforms.ToTensor(),                       # PIL图 → [0,1]张量
    transforms.Normalize(mean=[0.5], std=[0.5])  # 归一化到 [-1,1]
])

dataset = datasets.CIFAR10(root='./data', train=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=32)  # DataLoader 只管凑 batch，不管 transform

# 训练集和测试集的 transform 通常不同
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 训练时做增强
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),                      # 测试时只归一化，不增强
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

### ToTensor() 的作用

`transforms.ToTensor()` 做两件事：
1. 把 PIL.Image 或 numpy array 转成 PyTorch 张量
2. 自动把像素值从 [0,255] 缩放到 [0,1]

### Normalize 归一化

In [ ]:
transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
# 作用：img = (img - mean) / std
# [0,1] 的图经过 Normalize(mean=0.5, std=0.5) 后 → [-1, 1]
# 目的是让数据分布更稳定，加速收敛

---

## 面试汇总

### Q1: DataLoader 的 shuffle 是在哪个层面做的？

在 **Sampler 层面**。shuffle=True 时，DataLoader 内部使用 RandomSampler，在样本层面打乱索引顺序。

### Q2: num_workers 设置过大的副作用？

- 内存占用高（每个 worker 独立拷贝数据）
- 进程创建开销大
- 适合 2~8

### Q3: Sampler 和 shuffle 有什么区别？

shuffle 是 DataLoader 参数，本质是在创建 RandomSampler。Sampler 决定遍历顺序，shuffle 决定是否打乱。

### Q4: collate_fn 什么时候需要自定义？

- 变长序列（NLP）需要 padding
- 嵌套结构（检测任务）不能 stack
- 非标准数据格式

### Q5: 为什么数据增强要放在 `__getitem__` 里而不是 collate_fn 里？

collate_fn 在主进程串行执行，数据增强是 CPU 密集操作，会阻塞主进程导致 GPU 空等。`__getitem__` 在 worker 进程并行执行，增强和 GPU 训练可流水线并行。

### Q6: IterableDataset 和 Map-style Dataset 在多 worker 场景下有什么区别？

Map-style 的 sampler 索引会被复制到各 worker，按步长跳取不会重复。IterableDataset 每个 worker 都重新执行 `__iter__()`，会导致数据重复，必须通过 `worker_init_fn` 为各 worker 分配不同数据分片。

### Q7: pin_memory 为什么能加速？

锁页内存不会被换出到磁盘，CPU→GPU 传输可通过 DMA 直传，无需经过临时缓冲区拷贝。普通内存在传输前可能需要先换页回内存，再拷贝到临时锁页缓冲区，再传 GPU，多两步开销。

### Q8: shuffle 可以给 IterableDataset 用吗？

不可以。shuffle 依赖 `__len__` 构建随机索引，IterableDataset 没有长度，PyTorch 直接抛 `TypeError: shuffle option is not supported with IterableDataset`。流式数据的打乱需在数据库查询层面用 `ORDER BY RANDOM()` 实现。

---

## 代码练习

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# 变长序列的 collate_fn
def collate_fn(batch):
    texts, labels = zip(*batch)
    # Padding 到相同长度
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return padded_texts, torch.tensor(labels)

# 不平衡数据集的 WeightedRandomSampler
labels = torch.randint(0, 2, (1000,)).tolist()
class_counts = [sum(1 for l in labels if l == 0), sum(1 for l in labels if l == 1)]
weights = [1.0 / class_counts[l] for l in labels]
sampler = WeightedRandomSampler(weights, len(weights))

dataloader = DataLoader(
    TextDataset(texts, labels),
    batch_size=32,
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

for batch_texts, batch_labels in dataloader:
    print(batch_texts.shape, batch_labels.shape)

---

## 参考资料

- PyTorch DataLoader: https://pytorch.org/docs/stable/data.html
- Sampler: https://pytorch.org/docs/stable/data.html#data-samplers

# 六、模型保存与加载

---

## 6-1 state_dict 方式

### 模型结构 vs 模型参数

```
model = MyModel()     # 模型结构：类定义 + forward 逻辑
model.weight          # 模型参数：具体数值（tensor）
model.bias
model.running_mean    # BatchNorm 的缓存（也是 state_dict 的一部分）
```

**`model.state_dict()` 返回什么？**

In [ ]:
model.state_dict()
# OrderedDict([
#     ('layer1.weight', tensor([...])),
#     ('layer1.bias', tensor([...])),
#     ('layer2.weight', tensor([...])),
#     ...
# ])

只包含：**参数值 + Buffer（running_mean/var 等缓存）**，不包含类定义。

**state_dict 包含的是什么？**

In [ ]:
model.state_dict().keys()
# 包含两类：
# - layer.weight, layer.bias     ← nn.Parameter（可训练参数）
# - bn.running_mean/var         ← Buffer（不可训练，但会保存和加载）

> 注意：`model.parameters()` 和 `model.state_dict()` 不一样。前者只包含可训练参数，后者包含参数 + Buffer。

**为什么要保存 Buffer？**
- `running_mean/var` 是 BatchNorm 在训练时积累的统计量，推理时必须用
- 加载时如果不恢复这些值，BatchNorm 的行为会错乱

---

### torch.load vs load_state_dict（容易混淆）

这是**两个完全不同的操作**，经常被搞混：

In [ ]:
# torch.load()：从磁盘文件反序列化为 Python 对象
state = torch.load('checkpoint.pt', weights_only=True)
# state 的类型：dict（包含 model_state_dict、optimizer_state_dict、epoch 等）

# load_state_dict()：把字典里的 tensor 塞进模型参数
model.load_state_dict(state['model_state_dict'])
optimizer.load_state_dict(state['optimizer_state_dict'])

| 方法 | 作用 | 层级 |
|------|------|------|
| `torch.load(path)` | 读文件，返回 Python 对象 | 文件 → 内存 |
| `load_state_dict(dict)` | 把 dict 里的值赋给模型参数 | dict → 模型 |

**典型流程：**

In [ ]:
# 1. 加载 checkpoint 文件
checkpoint = torch.load('checkpoint.pt', weights_only=True)

# 2. 把各项塞回对应的地方
model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])
epoch = checkpoint['epoch']

---

### weights_only 参数（安全加载）

In [ ]:
# 默认（旧行为）
state = torch.load('checkpoint.pt')
# 可能执行任意 Python 代码（pickle 反序列化风险）

# 推荐写法
state = torch.load('checkpoint.pt', weights_only=True)
# 只允许加载 tensor、数值、字符串、字典、列表等安全类型

**weights_only=True 允许什么？**
- tensor ✅
- int / float / str ✅
- dict / list / tuple ✅
- 自定义 Python 类实例 ❌（拒绝）
- lambda 函数 ❌（拒绝）

> Buffer（running_mean 等）是 tensor，不受此限制，正常加载。

**为什么需要这个参数？**
pickle 可以序列化任意 Python 对象，反序列化时可能执行恶意代码。`weights_only=True` 强制 checkpoint 只包含数值，规避这个风险。

> ⚠️ 如果 checkpoint 里保存了自定义对象，用 `weights_only=True` 会报错。**建议永远不要在 checkpoint 里保存自定义对象。**

---

### Config 方式保存模型结构（工业标准）

模型结构和权重**分开存**，这是工业界的主流做法。

**以 HuggingFace 为例：**
```
model/
├── config.json      # 模型结构：层数、隐藏维度、注意力头数...
└── model.safetensors # 模型权重（state_dict）
```

**config.json 例子：**
```json
{
  "architectures": ["BertForMaskedLM"],
  "hidden_size": 768,
  "num_hidden_layers": 12,
  "num_attention_heads": 12,
  "intermediate_size": 3072,
  "vocab_size": 30522
}
```

**为什么不用 save 整个模型？**

| | save 整个模型 | save state_dict + config |
|---|---|---|
| 类定义 | ❌ 序列化在文件里，强依赖 | ✅ config.json 纯数据，任何系统都能读 |
| 部署兼容性 | ❌ Python pickle 环境可能不兼容 | ✅ 权重文件独立，任意推理引擎都能用 |
| 工具链支持 | ❌ 很多转换工具不支持 | ✅ ONNX/TensorRT/Triton 都基于 state_dict |

**自己实现一个 Config 类：**

In [ ]:
import json
import torch
import torch.nn as nn

class ModelConfig:
    def __init__(self, in_features=10, out_features=2, hidden=[64, 32]):
        self.in_features = in_features
        self.out_features = out_features
        self.hidden = hidden

    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.__dict__, f, indent=2)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls(**json.load(f))

class MLP(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        layers = []
        prev = config.in_features
        for h in config.hidden:
            layers.extend([nn.Linear(prev, h), nn.ReLU()])
            prev = h
        layers.append(nn.Linear(prev, config.out_features))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# 保存
config = ModelConfig(in_features=10, out_features=2, hidden=[64, 32])
config.save('config.json')
torch.save(model.state_dict(), 'model.pt')

# 加载（部署时：不需要源码，只需要 config.json + model.pt）
loaded_config = ModelConfig.load('config.json')
model = MLP(loaded_config)
model.load_state_dict(torch.load('model.pt', weights_only=True))

**工业流程：**
```
训练侧：save state_dict + config.json
         ↓
部署侧：读 config.json → 建模型结构 → load state_dict → 推理
```

---

### 为什么推荐 state_dict 而非整个模型

**保存整个模型（不推荐）**：

In [ ]:
torch.save(model, path)  # 把整个 model 对象序列化

- 依赖原始类定义，换个文件/类名就炸
- 包含整个 Python pickle 序列化环境，不干净
- 文件体积和 state_dict 几乎一样（差异只有几 MB，主要来自类定义）

**保存 state_dict（推荐）**：

In [ ]:
torch.save(model.state_dict(), path)  # 只保存参数

- 权重文件独立，推理引擎（ONNX/TensorRT/Triton）都能用
- 架构用 config.json 单独管理，部署更灵活
- 已经是工业标准，生态工具链都基于此
- 兼容性强，任何相同结构的模型都能加载
- 加载时需要自己先建模型，再塞参数

In [ ]:
# 保存
torch.save(model.state_dict(), 'model.pt')

# 加载（必须先有模型结构）
model = MyModel()                          # 先实例化结构
model.load_state_dict(torch.load('model.pt'))  # 再塞参数

---

### torch.save / torch.load 基础

In [ ]:
# 保存
torch.save(obj, path)          # obj 可以是 state_dict、整个 checkpoint、任意 Python 对象

# 加载
state = torch.load(path)       # 返回之前保存的对象

---

## 6-2 断点续训（Checkpoint）

### 为什么需要断点续训

训练中断（宕机、显存爆）后，从头重训代价大。Checkpoint 把训练状态全部存盘。

### checkpoint 包含的内容

In [ ]:
checkpoint = {
    'epoch': 5,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'scheduler': scheduler.state_dict(),     # 学习率调度器状态
    'scaler': amp_gradScaler.state_dict(),  # 混合精度状态
    'best_loss': 0.32,
}
torch.save(checkpoint, 'checkpoint_epoch5.pt')

| 字段 | 必须 | 说明 |
|------|------|------|
| model | ✅ | 模型参数 |
| optimizer | ✅ | 动量、学习率等状态 |
| epoch | 推荐 | 方便断点定位 |
| scheduler | 可选 | 学习率调度器状态 |
| scaler | 可选 | 混合精度训练状态 |
| best_loss | 可选 | 记录最优指标 |

### 完整恢复训练

In [ ]:
# 加载
checkpoint = torch.load('checkpoint_epoch5.pt')
model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])
scheduler.load_state_dict(checkpoint['scheduler'])
scaler.load_state_dict(checkpoint['scaler'])

start_epoch = checkpoint['epoch'] + 1
best_loss = checkpoint['best_loss']

# 继续训练
for epoch in range(start_epoch, num_epochs):
    train(...)
    validate(...)
    scheduler.step()
    save_checkpoint(...)

### EMA 的保存与加载

**EMA（指数移动平均）** 维护一套独立于主模型的参数副本，训练时用 EMA 参数做验证通常效果更好。

**EMA 参数是怎么更新的？**

In [ ]:
ema_model = AveragedModel(model)  # 包装原始模型，得到一套参数副本

# 每次 optimizer 更新完原模型参数后，立即更新 EMA 副本
ema_model.update_parameters(model)

# 内部公式：
# EMA_w = decay * 旧EMA_w + (1 - decay) * 当前权重
# 默认 decay = 0.999

**为什么是"双重动量"？**

| 动量 | 作用对象 | 目的 |
|------|---------|------|
| Optimizer momentum | 梯度更新方向 | 加速收敛、减少振荡 |
| EMA decay | 参数值本身 | 平滑参数曲线、抗过拟合 |

optimizer momentum 让收敛更快，但收敛路径可能有振荡。EMA 在参数层面再做一次平滑，取"均值"而非"终点"，往往泛化更好。

**参数完全一一对应，大小相同：**

In [ ]:
print("原模型参数量:", sum(p.numel() for p in model.parameters()))
# 55

print("EMA 参数量:", sum(p.numel() for p in ema_model.parameters()))
# 55（完全一样，EMA 是独立的副本）

# 所以 checkpoint = model + ema = 约 2 倍体积

**完整训练示例（含 AMP 混合精度 + EMA + Checkpoint）：**

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel

# 初始化
model = MyModel().cuda()
ema_model = AveragedModel(model)          # EMA 副本
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = GradScaler()                      # AMP 缩放器
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.CrossEntropyLoss()

def save_checkpoint(epoch, best_loss):
    checkpoint = {
        'epoch': epoch,
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),        # EMA 参数单独保存
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),       # 混合精度状态
        'best_loss': best_loss,
    }
    torch.save(checkpoint, f'checkpoint_epoch{epoch}.pt')

def load_checkpoint(path):
    checkpoint = torch.load(path, weights_only=True)
    model.load_state_dict(checkpoint['model'])
    ema_model.load_state_dict(checkpoint['ema'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    return checkpoint['epoch'], checkpoint['best_loss']

# 训练循环
best_loss = float('inf')
start_epoch = 0

for epoch in range(start_epoch, 30):
    model.train()
    for batch in dataloader:
        optimizer.zero_grad()

        # 混合精度前向
        with autocast():
            output = model(batch['input'])
            loss = criterion(output, batch['target'])

        # 混合精度反向
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # 更新 EMA 副本（在 optimizer 更新参数之后立即调用）
        ema_model.update_parameters(model)

    scheduler.step()

    # 验证（用 EMA 模型效果更好）
    model.eval()
    ema_model.eval()
    val_loss = validate(ema_model, val_loader)  # ← 用 EMA 模型验证

    if val_loss < best_loss:
        best_loss = val_loss
        save_checkpoint(epoch, best_loss)

# 加载 EMA 恢复训练
epoch, best_loss = load_checkpoint('checkpoint_epoch5.pt')
print(f"从 epoch {epoch} 恢复训练，最优 loss: {best_loss}")

**EMA 加载后的用法：**

In [ ]:
# 验证/推理时用 EMA 模型
ema_model.eval()
with torch.no_grad():
    output = ema_model(input)

# 继续训练时用普通模型（EMA 只用于推理）

---

## 6-3 部分加载（strict=False / key 过滤）

### strict=False 的完整加载逻辑

`strict=False` 按 key 名称一一匹配，规则只有三条：

| 情况 | 行为 |
|------|------|
| 两边 key 名称一样 | ✅ 加载进去 |
| 新模型有、pretrain 没有 | 保持随机初始化（不更新） |
| pretrain 有、新模型没有 | 直接丢弃 |

In [ ]:
# pretrain: L1, L2, L3
# new model: L1, L2, L4, L3

# strict=False 加载后：
# L1 → ✅ 加载
# L2 → ✅ 加载
# L3 → ✅ 加载（名称相同）
# L4 → 保持随机初始化（pretrain 没有）
# (pretrain 的其他层 → 丢弃)

### 迁移学习场景

预训练模型有 100 层，目标任务只需要前 90 层，后 10 层随机初始化：

In [ ]:
# 加载预训练权重
pretrained_dict = torch.load('pretrained.pt', weights_only=True)

# 只取前90层的权重（按名称前缀过滤）
filtered_dict = {k: v for k, v in pretrained_dict.items()
                  if k in model.state_dict() and k.startswith('layer')}
model.load_state_dict(filtered_dict, strict=False)

### 形状不匹配时（key 存在但 shape 不同）

In [ ]:
# pretrained: layer4.weight shape = (64, 512)
# new model:  layer4.weight shape = (64, 256)  ← 输出维度变了

# strict=False 碰到这种情况：
# → 跳过 layer4.weight（形状不一致，不会强制加载）
# → layer4 保持随机初始化

model.load_state_dict(pretrained, strict=False)

> 注意：`strict=False` 只解决"有没有这个 key"的问题，不解决"key 有但 shape 不对"的问题。

### key 过滤示例

In [ ]:
# 场景：去掉最后的分类头，只保留特征提取部分
pretrained_dict = torch.load('pretrained.pt', weights_only=True)
filtered_dict = {k: v for k, v in pretrained_dict.items() if 'classifier' not in k}
model.load_state_dict(filtered_dict, strict=False)

### 手动映射不同名称的层

如果两个模型 key 名称不同但想手动对应，需要改 pretrained 的 key：

In [ ]:
pretrained = torch.load('pretrained.pt', weights_only=True)

# 把 pretrained 里的 L3 → L4（手动重命名）
pretrained_renamed = {}
for k, v in pretrained.items():
    if k.startswith('L3'):
        new_key = k.replace('L3', 'L4')
        pretrained_renamed[new_key] = v
    else:
        pretrained_renamed[k] = v

model.load_state_dict(pretrained_renamed, strict=False)

---

## 6-4 跨设备保存与加载

### map_location 原理

`torch.load` 默认在保存的设备上加载。如果 GPU 显存不够，CPU 加载后转 GPU。

In [ ]:
# 保存时在 GPU 上
torch.save(model.state_dict(), 'model.pt')

# 加载时在 CPU 上（显存不够时）
state_dict = torch.load('model.pt', map_location='cpu')
model.load_state_dict(state_dict)

# 指定具体设备
state_dict = torch.load('model.pt', map_location='cuda:1')  # 加载到第2张卡

**常见设备组合：**

In [ ]:
# GPU训练 → CPU加载（用于部署/推理）
state = torch.load('model.pt', map_location='cpu')

# CPU训练 → GPU加载
state = torch.load('model.pt', map_location='cuda:0')

# GPU1 保存 → GPU0 加载
torch.save(model.state_dict(), 'model.pt')  # 在 GPU1 上保存
state = torch.load('model.pt', map_location='cuda:0')  # 在 GPU0 上加载

### 保存时 train / eval 模式重要吗？

**训练/推理的断点续训：模式不重要。** 因为保存的是参数值，`model.train()` 和 `model.eval()` 改变的是**层的统计行为**，不改变参数本身的数值。

**真正重要的是：什么时候保存？**

In [ ]:
# ✅ 正确：在 epoch 结束时保存
for epoch in range(num_epochs):
    model.train()
    for batch in dataloader:
        train_step(batch)  # running_mean/var 持续更新，越来越稳定
    
    # epoch 结束后保存：running_mean/var 已积累大量 batch，稳定
    torch.save({'model': model.state_dict(), ...}, f'ckpt_epoch{epoch}.pt')

# ❌ 错误：只训了几个 batch 就保存
for i, batch in enumerate(dataloader):
    train_step(batch)
    if i == 10:  # 只训了10个batch就跑路
        torch.save(model.state_dict(), 'ckpt.pt')  # running_mean/var 只基于10个batch，严重不稳定

**结论：**
- 断点续训以 **Epoch** 为切分点，running_mean/var 质量由"训练了多少步"决定，和模式无关
- 断点续训时，保存时的模式不重要，加载后切回 `model.train()` 继续训练即可
- 中途保底备份（按 Step 保存）可以存，但只当"保险"，不用于推理

### 单卡保存 / 多卡加载 / DDP

**DDP 训练时，模型会被包装：**

In [ ]:
model = DistributedDataParallel(model)  # DDP 包装了一层

model.module  # ← 用 .module 访问被包装的原始模型

**多卡保存（去掉 DDP 包装）：**

In [ ]:
# DDP 训练后保存，必须用 .module 拿到原始模型
torch.save(model.module.state_dict(), 'model.pt')  # ✅ 正确：保存原始模型
torch.save(model.state_dict(), 'model.pt')          # ❌ 错误：存的是 DDP 包装层状态

# 单卡加载（任意设备）
model = MyModel()
model.load_state_dict(torch.load('model.pt', weights_only=True))

**单卡保存 → 多卡加载：**

In [ ]:
# 1. 单卡保存
torch.save(model.state_dict(), 'model.pt')

# 2. 多卡加载
model = MyModel()
model.load_state_dict(torch.load('model.pt', weights_only=True))  # 先加载权重
model = DistributedDataParallel(model, device_ids=[0, 1, 2, 3])   # 再包 DDP
# 然后就可以多卡训练了

> 注意：权重文件和设备数量无关，同一个文件可以在单卡或多卡上用。


---

## 6-5 大模型与安全

### 分片保存（Sharded Checkpoint）

70B 参数的模型，单文件 `.pt` 可能超过 100GB，写入慢，加载更慢。分片保存把大文件拆成多个小文件。

**HuggingFace 分片保存：**

In [ ]:
model.save_pretrained('./model', max_shard_size='5GB')
# 生成：
# model-00001-of-00003.safetensors  (~5GB)
# model-00002-of-00003.safetensors  (~5GB)
# model-00003-of-00003.safetensors  (~剩余)
# config.json

**分片加载（核心就是循环读取所有分片，合并成一个 dict）：**

In [ ]:
from safetensors import safe_open

state_dict = {}
for shard_file in sorted(glob.glob('model-*-of-*.safetensors')):
    with safe_open(shard_file, framework="pt", device="cpu") as f:
        for key in f.keys():
            state_dict[key] = f.get_tensor(key)

# 合并后正常加载
model = MyModel()
model.load_state_dict(state_dict)

> 为什么用 safetensors 而不是 .pt？safetensors 是 HuggingFace 推出的安全格式，加载更快（无需反序列化，直接内存映射），且天然支持分片。

**PyTorch 原生分片（不常用）：**

In [ ]:
torch.save(model.state_dict(), 'model.pt',
           _use_new_zipfile_serialization=True)  # zipfile 格式，体积更小

### torch.load 的 weights_only 参数

In [ ]:
# 旧版本（默认行为）
state = torch.load('model.pt')  # 可能执行任意 Python 代码（pickle 反序列化风险）

# 新版本推荐
state = torch.load('model.pt', weights_only=True)  # 只加载 tensor，禁止执行代码

**weights_only=True 允许什么？**

| 类型 | 允许 |
|------|------|
| tensor、数值、字符串 | ✅ |
| dict / list / tuple | ✅ |
| 自定义 Python 类实例 | ❌ |
| lambda 函数 | ❌ |

> Buffer（running_mean 等）是 tensor，不受限制。

**为什么需要这个参数？**
pickle 可以序列化任意 Python 对象，反序列化时可能执行恶意代码。`weights_only=True` 强制 checkpoint 只包含数值。

**weights_only=True 报错怎么办？**

In [ ]:
# 原因：checkpoint 里保存了自定义对象（不推荐）
torch.load('model.pt', weights_only=True)
# RuntimeError: unsupported PyTorch type: __main__.MyCustomClass

# 解决：永远不要在 checkpoint 里保存自定义对象
checkpoint = {
    'model': model.state_dict(),    # ✅ 只有数值
    'optimizer': optimizer.state_dict(),  # ✅ 只有数值
    'epoch': 10,                   # ✅ int
    'config': {'lr': 1e-3},       # ✅ dict
    # 'preprocessor': MyPreprocessor()  ❌ 不要放自定义对象
}

---

## 面试汇总

### Q1: 模型保存推荐用哪种方式？为什么？

推荐 `torch.save(model.state_dict(), path)`，而不是 `torch.save(model, path)`。

state_dict 优势：
1. 权重文件独立，架构用 config.json 管理，部署到任意推理引擎（ONNX/TensorRT）都能用
2. 不依赖 Python pickle 环境，PyTorch 版本升级后仍能正常加载
3. 是 HuggingFace、Tim m 等主流生态的标准格式，工具链都基于此

### Q2: 断点续训需要保存哪些内容？

| 内容 | 必须 | 说明 |
|------|------|------|
| model state_dict | ✅ | 模型参数 |
| optimizer state_dict | ✅ | 动量、学习率状态 |
| epoch | 推荐 | 方便定位断点 |
| scheduler state_dict | 可选 | 学习率调度器状态 |
| scaler state_dict | 可选 | 混合精度状态 |
| EMA state_dict | 可选 | EMA 参数副本 |

核心是模型参数和优化器状态，其余都是辅助恢复。

### Q3: strict=False 和 key 过滤的区别？

| 方式 | 作用 |
|------|------|
| `strict=False` | 不匹配的 key 跳过（名称不一致、形状不一致都跳过） |
| key 过滤 | 主动筛选要加载的 key（如去掉分类头） |
| 两者结合 | 先过滤再加载 |

### Q4: 为什么加载后要 model.eval()？

BatchNorm 和 Dropout 在 train/eval 模式下行为不同：

- BatchNorm：eval 模式用 **running_mean/var** 归一化；train 模式用 **batch 统计量**
- Dropout：eval 模式全部通过；train 模式随机置零

如果不设 eval()，BatchNorm 会用当前 batch 的统计量（可能只有1个样本），推理结果完全不稳定。

### Q5: map_location 是什么原理？

`torch.load` 默认在保存的设备加载。`map_location` 重新指定加载目标设备，底层是把 tensor 从原设备 copy 到新设备。

In [ ]:
# GPU1 保存 → GPU0 加载
torch.save(model.state_dict(), 'model.pt')  # GPU1
state = torch.load('model.pt', map_location='cuda:0')  # 加载到 GPU0

### Q6: weights_only=True 解决什么问题？

防止恶意 checkpoint 通过 pickle 反序列化执行任意代码。只允许加载 tensor 等安全类型。

### Q7: EMA 是什么？为什么用它？

EMA（指数移动平均）维护一套独立于主模型的参数副本，用 decay 公式平滑更新：
```
EMA_w = 0.999 * 旧EMA + 0.001 * 当前权重
```

效果：对训练过程的参数轨迹做二次平滑，取"均值"而非"终点"，泛化更好。验证/推理时用 EMA 参数。

### Q8: DDP 训练时保存为什么要用 .module？

`DistributedDataParallel` 包装了原始模型。直接 `model.state_dict()` 会保存 DDP 包装层状态，加载时会出错。用 `model.module.state_dict()` 才能拿到原始模型参数。

---

## 代码练习

In [ ]:
import torch
import torch.nn as nn

# ===== 1. 保存和加载 state_dict =====
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    
    def forward(self, x):
        return self.fc(x)

model = SimpleModel()

# 保存（推荐）
torch.save(model.state_dict(), 'model.pt')

# 加载
model2 = SimpleModel()
model2.load_state_dict(torch.load('model.pt', weights_only=True))

# ===== 2. 断点续训 =====
checkpoint = {
    'epoch': 5,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'best_loss': 0.32,
}
torch.save(checkpoint, 'checkpoint.pt')

# 加载恢复
ckpt = torch.load('checkpoint.pt', weights_only=True)
model.load_state_dict(ckpt['model'])
optimizer.load_state_dict(ckpt['optimizer'])
start_epoch = ckpt['epoch'] + 1

# ===== 3. 部分加载（迁移学习）=====
pretrained_dict = torch.load('pretrained.pt', weights_only=True)
model_dict = model.state_dict()

# 只加载匹配的层
pretrained_dict = {k: v for k, v in pretrained_dict.items()
                   if k in model_dict and v.shape == model_dict[k].shape}
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict, strict=False)

# ===== 4. 跨设备加载 =====
# GPU保存 → CPU加载
state = torch.load('model.pt', map_location='cpu')
model.load_state_dict(state)

# ===== 5. 单卡保存多卡加载 =====
# DDP训练时保存要去掉 .module 包装
torch.save(model.module.state_dict(), 'model.pt')

# 七、GPU 加速与分布式

| 知识点 | 说明 |
|---|---|
| 单 GPU | `model.cuda()` / `tensor.to(device)` |
| 多 GPU | `nn.DataParallel(model, device_ids=[0,1,2])` |
| 多 GPU 原理 | 按 batch 维度分割 → 各 GPU 独立 forward → 梯度累加到主 GPU |
| 分布式 DDP | `DistributedDataParallel` — 工业级多机多卡 |
| `torch.cuda.is_available()` | 判断 GPU 是否可用 |

---

## 7-1 单 GPU 迁移

## 7-2 DataParallel（DP）多卡

## 7-3 DistributedDataParallel（DDP）原理

## 7-4 多卡实验与验证

---

## 代码练习

# 八、Fine-tuning（微调）

| 方式 | 做法 |
|---|---|
| 局部微调 | 冻结底层参数（`requires_grad=False`），只训练顶层 |
| 全局微调 | 不同层设不同学习率（在 optimizer param_groups 中配置） |
| 加载预训练 | `model = torchvision.models.resnet18(pretrained=True)` |

---

## 8-1 预训练模型加载

## 8-2 冻结层微调（Frozen）

## 8-3 不同层不同学习率

## 8-4 Adapter 简介

---

## 代码练习

# 九、常见网络层速查

```
卷积:  nn.Conv2d(in, out, kernel, stride, padding)
池化:  nn.MaxPool2d / nn.AvgPool2d / nn.AdaptiveAvgPool2d
BN:    nn.BatchNorm2d(channels)  — train时用batch统计，eval时用全局统计
Dropout: nn.Dropout(p)           — train时随机置0，eval时全部保留
全连接: nn.Linear(in_features, out_features)
激活:  nn.ReLU / nn.Sigmoid / nn.Tanh
```

**卷积层参数说明：** in_channels / out_channels / kernel_size / stride / padding

---

## 9-1 卷积层（Conv2d）

## 9-2 池化层（MaxPool / AvgPool / AdaptiveAvgPool）

## 9-3 归一化层（BatchNorm / LayerNorm / InstanceNorm）

## 9-4 Dropout 与正则化

## 9-5 激活函数对比

## 9-6 常用组合

---

## 代码练习

# 十、训练流程全链路（必须能默写）

```
1. 定义 Dataset + DataLoader
2. 定义模型 (继承nn.Module) → 放到GPU
3. 定义损失函数 (CrossEntropyLoss / MSELoss...)
4. 定义优化器 (SGD / Adam)
5. 训练循环:
   for epoch in range(E):
       model.train()
       for batch in train_loader:
           optimizer.zero_grad()
           output = model(input)
           loss = criterion(output, target)
           loss.backward()
           optimizer.step()
       model.eval()
       with torch.no_grad():
           ...
```

---

## 10-1 完整训练循环

## 10-2 验证与测试流程

## 10-3 训练监控（Loss 曲线、Early Stopping）

## 10-4 常见问题与调试

---

## 代码练习